# EMCRAG-FT v07.01 — Data Integrity, Dataset Splits, SFT Export e Pilot Preparation

**Project:** Edge Medical Corrective RAG (EMCRAG)  
**Baseline congelata:** `CRAG_MedGemma_v.06.11`  
**Nuovo ramo sperimentale:** `EMCRAG-FT v07.x`  
**Notebook:** `EMCRAG_FT_v.07.01_DataPrep_Pilot.ipynb`

## Obiettivo

Questo notebook implementa le **Fasi 0–3** dello studio di fine-tuning di MedGemma 1.5 4B-it:

1. **Fase 0 — Experimental freeze**
   - registra la baseline EMCRAG v06.11;
   - congela path e parametri di split;
   - non modifica pipeline CRAG né benchmark RAGAS.

2. **Fase 1 — Data-integrity audit e split**
   - carica IMB-QA completo e `IMB_Curated_Test_Set_1000`;
   - verifica schema e consistenza;
   - esclude in modo verificabile il frozen test dal pool addestrabile;
   - genera DEV, VALIDATION e TRAIN POOL senza sovrapposizioni;
   - preserva la distribuzione delle `General Category` per DEV/VALIDATION;
   - prepara un ordine TRAIN deterministico e annidato.

3. **Fase 2 — Conversione SFT**
   - produce record `messages` System/User/Assistant;
   - non parafrasa né riscrive Question/Answer;
   - misura la lunghezza in token col tokenizer MedGemma;
   - prepara il dataset per Unsloth Studio.

4. **Fase 3 — Pilot locale**
   - esporta `PILOT-1K`;
   - produce una configurazione QLoRA iniziale;
   - verifica GPU/BF16;
   - **non esegue il training**: il pilot viene lanciato in Unsloth Studio.

## Principio metodologico

Il test congelato non deve essere usato per training, validation, model selection, scelta degli iperparametri o scelta della dimensione del dataset.

Il benchmark RAGAS/GPT-4o rimane riservato alla fase finale.


## Nota metodologica importante sul `source_index` della v06.11

Nella v06.11 il campo `metadata["source_index"] = idx` viene creato durante la normalizzazione del curated dataset. È quindi un indice posizionale nel curated set, non una chiave originaria affidabile del dataset IMB-QA completo.

Per questo notebook:

- **Metodo A:** usa l'`ID` originale solo se presente sia nel dataset completo sia nel frozen test e se il contenuto Question+Answer è coerente.
- **Metodo B:** usa un hash SHA-256 deterministico di `Question + Answer`, dopo normalizzazione conservativa Unicode NFKC + whitespace.
- Il fuzzy matching non viene usato automaticamente.
- Se un record congelato non viene risolto con certezza, la costruzione degli split si interrompe.
- Se un frozen Question+Answer compare più volte nel full dataset, tutte le copie identiche vengono escluse.


In [ ]:
# ======================================================================================
# CELL 1 — IMPORTS E CONFIGURAZIONE DELLO STUDIO
# ======================================================================================

from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Optional, Iterable
import hashlib
import json
import os
import re
import unicodedata
import platform
import sys
import time

import numpy as np
import pandas as pd
from datasets import load_from_disk, Dataset, DatasetDict

STUDY_VERSION = "v07.01"
BASELINE_VERSION = "CRAG_MedGemma_v.06.11"
FT_STUDY_NAME = "EMCRAG-FT"

# Windows: F:\processed_datasets\...  -> WSL2: /mnt/f/processed_datasets/...
IMB_FULL_PATH = Path("/mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe")
IMB_FROZEN_TEST_PATH = Path("/mnt/f/processed_datasets/IMB_Curated_Test_Set_1000")

PROJECT_ROOT = Path.home() / "Project_CRAG_MedGemma"
FT_ROOT = PROJECT_ROOT / "fine_tuning" / STUDY_VERSION.replace(".", "_")

DIR_MANIFESTS = FT_ROOT / "manifests"
DIR_DATASETS = FT_ROOT / "datasets"
DIR_REPORTS = FT_ROOT / "reports"
DIR_PILOT = FT_ROOT / "pilot"
DIR_EXPORTS = FT_ROOT / "exports"

for p in [DIR_MANIFESTS, DIR_DATASETS, DIR_REPORTS, DIR_PILOT, DIR_EXPORTS]:
    p.mkdir(parents=True, exist_ok=True)

FT_SPLIT_SEED = 20260831
DEV_SIZE = 1_000
VALIDATION_SIZE = 2_000
PILOT_SIZE = 1_000
LEARNING_CURVE_SIZES = [5_000, 10_000, 25_000, 50_000, 100_000]

DEDUPLICATE_QA = True

# None = audit sull'intero train pool; 20k = stima rapida e riproducibile.
TOKEN_STATS_SAMPLE_SIZE = 20_000

MEDGEMMA_MODEL_ID = "google/medgemma-1.5-4b-it"
HF_HOME = Path("/mnt/f/hf_home_cache")
os.environ.setdefault("HF_HOME", str(HF_HOME))

SYSTEM_PROMPT_SFT = (
    "Sei un assistente medico che risponde in lingua italiana. "
    "Fornisci una risposta pertinente e chiara alla domanda dell'utente."
)

# Dati documentali da verificare, non assunti come ground truth.
README_EXPECTED_ROWS = 783_192
PLOT_CATEGORY_TOTAL = 783_092

EXPECTED_FULL_COLUMNS = [
    "ID", "General Category", "Category", "Data", "Url", "Question", "Answer"
]

print("Study:", STUDY_VERSION)
print("Baseline:", BASELINE_VERSION)
print("FT root:", FT_ROOT)
print("IMB full:", IMB_FULL_PATH)
print("Frozen test:", IMB_FROZEN_TEST_PATH)


: 

In [2]:
# ======================================================================================
# CELL 2 — FUNZIONI GENERALI PER DATASET HF ARROW
# ======================================================================================

def load_single_dataset(path: Path, preferred_split: Optional[str] = None) -> Dataset:
    """Carica un Dataset HF salvato su disco e risolve DatasetDict se univoco."""
    if not path.exists():
        raise FileNotFoundError(f"Path non trovato: {path}")

    obj = load_from_disk(str(path))

    if isinstance(obj, Dataset):
        return obj

    if isinstance(obj, DatasetDict):
        if preferred_split and preferred_split in obj:
            return obj[preferred_split]
        if len(obj) == 1:
            return next(iter(obj.values()))
        raise ValueError(
            f"{path} contiene più split {list(obj.keys())}; specificare preferred_split."
        )

    raise TypeError(f"Tipo dataset non supportato: {type(obj)}")


def resolve_column(columns: List[str], candidates: List[str], required: bool = True) -> Optional[str]:
    """Ricerca case-insensitive del primo nome candidato presente."""
    lower_map = {c.lower(): c for c in columns}
    for c in candidates:
        if c in columns:
            return c
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    if required:
        raise KeyError(f"Nessuna colonna tra {candidates}. Presenti: {columns}")
    return None


def describe_dataset(name: str, ds: Dataset) -> None:
    print("=" * 100)
    print(name)
    print("=" * 100)
    print("Rows    :", len(ds))
    print("Columns :", len(ds.column_names))
    print("Names   :", ds.column_names)
    print("Features:", ds.features)
    print()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


In [3]:
# ======================================================================================
# CELL 3 — FASE 0: CARICAMENTO E SCHEMA AUDIT
# ======================================================================================

full_ds = load_single_dataset(IMB_FULL_PATH)
frozen_ds = load_single_dataset(IMB_FROZEN_TEST_PATH)

describe_dataset("IMB-QA FULL", full_ds)
describe_dataset("IMB FROZEN TEST", frozen_ds)

missing_expected = [c for c in EXPECTED_FULL_COLUMNS if c not in full_ds.column_names]
extra_columns = [c for c in full_ds.column_names if c not in EXPECTED_FULL_COLUMNS]

print("Schema audit FULL")
print("  Missing expected columns:", missing_expected)
print("  Extra columns           :", extra_columns)

if missing_expected:
    raise RuntimeError(
        "Schema FULL diverso dal README. Chiarire prima di proseguire."
    )

FROZEN_Q_COL = resolve_column(
    frozen_ds.column_names,
    ["Question", "question", "domanda", "query", "input", "text", "prompt"]
)
FROZEN_A_COL = resolve_column(
    frozen_ds.column_names,
    ["Answer", "answer", "reference_answer", "risposta", "ground_truth", "target"]
)
FROZEN_ID_COL = resolve_column(
    frozen_ds.column_names,
    ["ID", "id", "source_id", "record_id"],
    required=False
)

print("\nFrozen column mapping")
print("  Question:", FROZEN_Q_COL)
print("  Answer  :", FROZEN_A_COL)
print("  ID      :", FROZEN_ID_COL)

if len(frozen_ds) != 1000:
    raise RuntimeError(
        f"Frozen test atteso = 1000; trovato = {len(frozen_ds)}. STOP."
    )


IMB-QA FULL
Rows    : 783192
Columns : 7
Names   : ['ID', 'General Category', 'Category', 'Data', 'Url', 'Question', 'Answer']
Features: {'ID': Value('int64'), 'General Category': Value('string'), 'Category': Value('string'), 'Data': Value('string'), 'Url': Value('string'), 'Question': Value('string'), 'Answer': Value('string')}

IMB FROZEN TEST
Rows    : 1000
Columns : 5
Names   : ['ID', 'Question', 'Answer', 'General Category', 'Category']
Features: {'ID': Value('int64'), 'Question': Value('string'), 'Answer': Value('string'), 'General Category': Value('string'), 'Category': Value('string')}

Schema audit FULL
  Missing expected columns: []
  Extra columns           : []

Frozen column mapping
  Question: Question
  Answer  : Answer
  ID      : ID


In [4]:
# ======================================================================================
# CELL 4 — CONSISTENCY AUDIT: README 783192 vs GRAFICO 783092
# ======================================================================================

print("=" * 100)
print("DATASET SIZE CONSISTENCY")
print("=" * 100)
print(f"Observed full rows         : {len(full_ds):,}")
print(f"README documented rows     : {README_EXPECTED_ROWS:,}")
print(f"Attached plot total        : {PLOT_CATEGORY_TOTAL:,}")
print(f"Observed - README          : {len(full_ds) - README_EXPECTED_ROWS:+,}")
print(f"Observed - plot total      : {len(full_ds) - PLOT_CATEGORY_TOTAL:+,}")

general_categories = pd.Series(full_ds["General Category"], dtype="string")
normalized_gc = general_categories.fillna("").str.strip()
missing_general_category = int((normalized_gc == "").sum())

category_counts = (
    normalized_gc.replace("", "__MISSING__")
    .value_counts(dropna=False)
)

print("\nGeneral Category counts")
display(category_counts.rename("count").to_frame())

print("\nMissing/blank General Category:", missing_general_category)
print("Sum of observed categories:", int(category_counts.sum()))

category_counts.rename_axis("General Category").reset_index(name="count").to_csv(
    DIR_REPORTS / "full_general_category_distribution.csv",
    index=False
)

if len(full_ds) == README_EXPECTED_ROWS and PLOT_CATEGORY_TOTAL != len(full_ds):
    print(
        "\nATTENZIONE: dataset osservato = README ma grafico differisce di "
        f"{len(full_ds) - PLOT_CATEGORY_TOTAL} record. "
        "Verificare categorie mancanti/escluse dal grafico."
    )
elif len(full_ds) != README_EXPECTED_ROWS:
    print(
        "\nATTENZIONE: row count osservata diversa dal README. "
        "La misura osservata verrà registrata nel manifest."
    )


DATASET SIZE CONSISTENCY
Observed full rows         : 783,192
README documented rows     : 783,192
Attached plot total        : 783,092
Observed - README          : +0
Observed - plot total      : +100

General Category counts


,count
"Urologia, Andrologia e Salute Maschile",110052
Gastroenterologia e Salute Digestiva,104449
Salute Mentale,103893
Medicina Generale e Chirurgia Generale,87789
"Oculistica, Otorinolaringoiatria, Odontoiatria e Pneumologia",83710
"Cardiologia, Sistema Circolatorio ed Ematologia",81232
Ginecologia e Salute Femminile,65792
Ortopedia e Sistema Muscoloscheletrico,50283
"Dermatologia, Allergie e Estetica",49288
Neurologia,46704



Missing/blank General Category: 0
Sum of observed categories: 783192

ATTENZIONE: dataset osservato = README ma grafico differisce di 100 record. Verificare categorie mancanti/escluse dal grafico.


## Fase 1 — Identificazione certa del frozen test dentro IMB-QA completo

### Metodo B: hash di contenuto

La normalizzazione è volutamente conservativa:

1. conversione a stringa;
2. Unicode NFKC;
3. whitespace multiplo → singolo spazio;
4. `strip()`.

Non vengono applicati lowercase/casefold, stemming, correzione ortografica, paraphrase o fuzzy matching.

La chiave è:

```text
SHA256(normalized(Question) + "\u241E" + normalized(Answer))
```

Ogni frozen record deve avere almeno un match esatto verificabile nel full dataset.

Se un frozen QA compare più volte nel full dataset, tutte le copie exact-content vengono escluse dal pool addestrabile.


In [5]:
# ======================================================================================
# CELL 5 — HASH FUNCTIONS E FROZEN KEYS
# ======================================================================================

RE_WS = re.compile(r"\s+")

def normalize_text_for_hash(value) -> str:
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    return RE_WS.sub(" ", text).strip()


def qa_hash(question, answer) -> str:
    q = normalize_text_for_hash(question)
    a = normalize_text_for_hash(answer)
    payload = q + "\u241E" + a
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


frozen_df = frozen_ds.to_pandas()
frozen_df["_frozen_row"] = np.arange(len(frozen_df), dtype=np.int64)
frozen_df["_qa_hash"] = [
    qa_hash(q, a)
    for q, a in zip(frozen_df[FROZEN_Q_COL], frozen_df[FROZEN_A_COL])
]

if FROZEN_ID_COL:
    frozen_df["_frozen_id"] = frozen_df[FROZEN_ID_COL].astype("string")
else:
    frozen_df["_frozen_id"] = pd.Series([pd.NA] * len(frozen_df), dtype="string")

print("Frozen rows:", len(frozen_df))
print("Unique frozen QA hashes:", frozen_df["_qa_hash"].nunique())
print("Duplicate frozen QA hashes:", len(frozen_df) - frozen_df["_qa_hash"].nunique())

if FROZEN_ID_COL:
    print("Unique frozen IDs:", frozen_df["_frozen_id"].nunique(dropna=True))

display(
    frozen_df[
        ["_frozen_row", "_frozen_id", FROZEN_Q_COL, FROZEN_A_COL, "_qa_hash"]
    ].head(5)
)


In [6]:
# ======================================================================================
# CELL 6 — FULL DATASET SCAN: METHOD A + METHOD B + METADATA MINIMA
# ======================================================================================
# Scansiona il full dataset una sola volta.
# Non costruisce un DataFrame con Question/Answer completi.
# ======================================================================================

FULL_ID_COL = "ID"
FULL_Q_COL = "Question"
FULL_A_COL = "Answer"
FULL_GENERAL_CAT_COL = "General Category"

frozen_hash_set = set(frozen_df["_qa_hash"].tolist())
frozen_id_set = set(
    x for x in frozen_df["_frozen_id"].dropna().astype(str).tolist()
    if x and x != "<NA>"
)

hash_matches = defaultdict(list)
id_matches = defaultdict(list)

meta_row_index = []
meta_id = []
meta_general_category = []
meta_hash = []
meta_valid = []

BATCH_SIZE = 2_000
start = time.time()
global_row = 0

scan_ds = full_ds.select_columns(
    [FULL_ID_COL, FULL_GENERAL_CAT_COL, FULL_Q_COL, FULL_A_COL]
)

for batch_no, batch in enumerate(scan_ds.iter(batch_size=BATCH_SIZE), start=1):
    ids = batch[FULL_ID_COL]
    cats = batch[FULL_GENERAL_CAT_COL]
    questions = batch[FULL_Q_COL]
    answers = batch[FULL_A_COL]

    for i, (rid, cat, q, a) in enumerate(zip(ids, cats, questions, answers)):
        row_idx = global_row + i

        nq = normalize_text_for_hash(q)
        na = normalize_text_for_hash(a)
        valid = bool(nq and na)
        h = qa_hash(q, a) if valid else None
        rid_s = "" if rid is None else str(rid)

        meta_row_index.append(row_idx)
        meta_id.append(rid_s)
        meta_general_category.append("" if cat is None else str(cat))
        meta_hash.append(h)
        meta_valid.append(valid)

        if h in frozen_hash_set:
            hash_matches[h].append(row_idx)

        if frozen_id_set and rid_s in frozen_id_set:
            id_matches[rid_s].append(row_idx)

    global_row += len(ids)

    if batch_no % 50 == 0:
        elapsed = time.time() - start
        print(f"Scanned {global_row:,}/{len(full_ds):,} rows | {elapsed/60:.1f} min")

full_meta_df = pd.DataFrame({
    "row_index": np.asarray(meta_row_index, dtype=np.int64),
    "ID": pd.Series(meta_id, dtype="string"),
    "General Category": pd.Series(meta_general_category, dtype="string"),
    "qa_hash": pd.Series(meta_hash, dtype="string"),
    "valid_qa": np.asarray(meta_valid, dtype=bool),
})

elapsed = time.time() - start
print(f"\nFULL SCAN COMPLETED: {len(full_meta_df):,} rows in {elapsed/60:.2f} min")

full_meta_path = DIR_MANIFESTS / "imb_full_metadata_minimal.parquet"
full_meta_df.to_parquet(full_meta_path, index=False)
print("Saved:", full_meta_path)


Scanned 100,000/783,192 rows | 0.1 min
Scanned 200,000/783,192 rows | 0.2 min
Scanned 300,000/783,192 rows | 0.4 min
Scanned 400,000/783,192 rows | 0.5 min
Scanned 500,000/783,192 rows | 0.7 min
Scanned 600,000/783,192 rows | 0.8 min
Scanned 700,000/783,192 rows | 1.0 min

FULL SCAN COMPLETED: 783,192 rows in 1.02 min
Saved: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/manifests/imb_full_metadata_minimal.parquet


In [7]:
# ======================================================================================
# CELL 7 — LEAKAGE AUDIT: CONCORDANZA ID / CONTENT HASH
# ======================================================================================

# Map veloce row_index -> qa_hash
full_hash_by_row = dict(
    zip(full_meta_df["row_index"].astype(int), full_meta_df["qa_hash"].astype("string"))
)

audit_rows = []

for _, r in frozen_df.iterrows():
    frow = int(r["_frozen_row"])
    fh = r["_qa_hash"]
    fid = None if pd.isna(r["_frozen_id"]) else str(r["_frozen_id"])

    hash_rows = [int(x) for x in hash_matches.get(fh, [])]
    id_rows = [int(x) for x in id_matches.get(fid, [])] if fid else []

    id_content_verified_rows = [
        row_idx for row_idx in id_rows
        if str(full_hash_by_row.get(row_idx)) == str(fh)
    ]

    audit_rows.append({
        "frozen_row": frow,
        "frozen_id": fid,
        "qa_hash": fh,
        "hash_match_count": len(hash_rows),
        "hash_match_rows": hash_rows,
        "id_match_count": len(id_rows),
        "id_content_verified_count": len(id_content_verified_rows),
        "id_verified_rows": id_content_verified_rows,
        "resolved": (len(hash_rows) > 0) or (len(id_content_verified_rows) > 0),
    })

leakage_audit_df = pd.DataFrame(audit_rows)

print("=" * 100)
print("FROZEN TEST LEAKAGE AUDIT")
print("=" * 100)
print("Frozen rows                     :", len(leakage_audit_df))
print("Resolved                        :", int(leakage_audit_df["resolved"].sum()))
print("Unresolved                      :", int((~leakage_audit_df["resolved"]).sum()))
print("Resolved by exact Q+A hash      :", int((leakage_audit_df["hash_match_count"] > 0).sum()))
print("Duplicated Q+A in full          :", int((leakage_audit_df["hash_match_count"] > 1).sum()))
print("Verified raw ID + same content  :", int((leakage_audit_df["id_content_verified_count"] > 0).sum()))

METHOD_A_AVAILABLE = FROZEN_ID_COL is not None
print("Method A raw ID available       :", METHOD_A_AVAILABLE)

leakage_audit_path = DIR_REPORTS / "frozen_test_leakage_audit.parquet"
leakage_audit_df.to_parquet(leakage_audit_path, index=False)

unresolved_df = leakage_audit_df[~leakage_audit_df["resolved"]].copy()

if len(unresolved_df):
    unresolved_path = DIR_REPORTS / "frozen_test_UNRESOLVED.csv"
    unresolved_df.to_csv(unresolved_path, index=False)
    display(unresolved_df.head(20))
    raise RuntimeError(
        f"STOP: {len(unresolved_df)} frozen record non risolti. "
        f"Vedi {unresolved_path}. Non creare gli split."
    )

print("\nOK: tutti i frozen record sono stati risolti con match esatto verificabile.")


FROZEN TEST LEAKAGE AUDIT
Frozen rows                     : 1000
Resolved                        : 1000
Unresolved                      : 0
Resolved by exact Q+A hash      : 1000
Duplicated Q+A in full          : 0
Verified raw ID + same content  : 1000
Method A raw ID available       : True

OK: tutti i frozen record sono stati risolti con match esatto verificabile.


In [8]:
# ======================================================================================
# CELL 8 — INSIEME DI ESCLUSIONE FROZEN TEST
# ======================================================================================

frozen_exclusion_rows = set()

for rows in leakage_audit_df["hash_match_rows"]:
    frozen_exclusion_rows.update(int(x) for x in rows)

for rows in leakage_audit_df["id_verified_rows"]:
    frozen_exclusion_rows.update(int(x) for x in rows)

print("Frozen logical records       :", len(frozen_ds))
print("Full rows excluded for test  :", len(frozen_exclusion_rows))
print("Extra excluded exact copies  :", len(frozen_exclusion_rows) - len(frozen_ds))

if len(frozen_exclusion_rows) < len(frozen_ds):
    raise RuntimeError(
        "L'insieme di esclusione è più piccolo del frozen test: audit non valido."
    )


Frozen logical records       : 1000
Full rows excluded for test  : 1000
Extra excluded exact copies  : 0


## Strategia di split

Dopo l'esclusione del frozen test:

1. vengono esclusi Question/Answer vuoti;
2. opzionalmente vengono eliminati i duplicati esatti Question+Answer;
3. DEV = 1.000 record stratificati per `General Category`;
4. VALIDATION = 2.000 record stratificati sul residuo;
5. TRAIN POOL = tutto il resto;
6. il TRAIN POOL riceve un ordine deterministico;
7. i subset usano sempre i primi N dello stesso ordine:

```text
PILOT-1K ⊂ FT-5K ⊂ FT-10K ⊂ FT-25K ⊂ FT-50K ⊂ FT-100K
```


In [9]:
# ======================================================================================
# CELL 9 — FUNZIONI DI SPLIT STRATIFICATO E ORDINE NESTED
# ======================================================================================

def stable_int_seed(text: str, base_seed: int) -> int:
    h = hashlib.sha256(f"{base_seed}|{text}".encode("utf-8")).hexdigest()
    return int(h[:8], 16)


def allocate_quotas(counts: pd.Series, n: int) -> Dict[str, int]:
    """Largest-remainder proportional allocation."""
    counts = counts.astype(int)
    if n > counts.sum():
        raise ValueError("n maggiore della popolazione disponibile.")

    raw = counts / counts.sum() * n
    base = np.floor(raw).astype(int)
    remainder = n - int(base.sum())
    frac = (raw - base).sort_values(ascending=False)
    quotas = base.to_dict()

    for cat in frac.index[:remainder]:
        quotas[cat] += 1

    for cat in quotas:
        quotas[cat] = min(quotas[cat], int(counts.loc[cat]))

    missing = n - sum(quotas.values())
    if missing > 0:
        spare = (counts - pd.Series(quotas)).sort_values(ascending=False)
        for cat, avail in spare.items():
            while missing > 0 and avail > 0:
                quotas[cat] += 1
                avail -= 1
                missing -= 1
            if missing == 0:
                break

    assert sum(quotas.values()) == n
    return quotas


def stratified_sample_rows(df: pd.DataFrame, n: int, seed: int) -> np.ndarray:
    temp = df.copy()
    temp["_stratum"] = (
        temp["General Category"]
        .fillna("__MISSING__")
        .replace("", "__MISSING__")
        .astype(str)
    )

    counts = temp["_stratum"].value_counts()
    quotas = allocate_quotas(counts, n)

    selected = []
    for cat, q in quotas.items():
        group = temp[temp["_stratum"] == cat]
        rs = stable_int_seed(cat, seed)
        selected.extend(
            group.sample(n=q, random_state=rs, replace=False)["row_index"]
            .astype(int).tolist()
        )

    rng = np.random.default_rng(seed)
    selected = np.asarray(selected, dtype=np.int64)
    rng.shuffle(selected)

    assert len(selected) == n
    assert len(np.unique(selected)) == n
    return selected


def nested_order_key(row_index: int, record_id: str, seed: int) -> str:
    return hashlib.sha256(
        f"{seed}|{row_index}|{record_id}".encode("utf-8")
    ).hexdigest()


In [10]:
# ======================================================================================
# CELL 10 — CREAZIONE DEV / VALIDATION / TRAIN POOL
# ======================================================================================

meta = full_meta_df.copy()
meta["split"] = "candidate"

# 1. Frozen exclusions
meta.loc[meta["row_index"].isin(frozen_exclusion_rows), "split"] = "frozen_test_excluded"

# 2. Invalid Q/A
meta.loc[
    (meta["split"] == "candidate") & (~meta["valid_qa"]),
    "split"
] = "invalid_qa"

# 3. Exact duplicate Q+A removal, solo sul pool addestrabile
if DEDUPLICATE_QA:
    candidate_mask = meta["split"] == "candidate"
    candidate_part = meta.loc[candidate_mask]
    duplicate_mask = candidate_part.duplicated(subset=["qa_hash"], keep="first")
    duplicate_rows = candidate_part.loc[duplicate_mask, "row_index"].astype(int)
    meta.loc[meta["row_index"].isin(duplicate_rows), "split"] = "duplicate_removed"

candidate_df = meta[meta["split"] == "candidate"].copy()
print("Candidate after exclusion/cleaning:", len(candidate_df))

# DEV
dev_rows = stratified_sample_rows(candidate_df, DEV_SIZE, FT_SPLIT_SEED + 1)
meta.loc[meta["row_index"].isin(dev_rows), "split"] = "dev"

# VALIDATION sul residuo
candidate_after_dev = meta[meta["split"] == "candidate"].copy()
val_rows = stratified_sample_rows(
    candidate_after_dev,
    VALIDATION_SIZE,
    FT_SPLIT_SEED + 2
)
meta.loc[meta["row_index"].isin(val_rows), "split"] = "validation"

# TRAIN POOL
meta.loc[meta["split"] == "candidate", "split"] = "train_pool"

train_mask = meta["split"] == "train_pool"
train_order_keys = [
    nested_order_key(
        int(row_idx),
        "" if pd.isna(rid) else str(rid),
        FT_SPLIT_SEED
    )
    for row_idx, rid in zip(
        meta.loc[train_mask, "row_index"],
        meta.loc[train_mask, "ID"]
    )
]
meta.loc[train_mask, "train_order_key"] = train_order_keys

train_order_df = (
    meta.loc[
        train_mask,
        ["row_index", "ID", "General Category", "qa_hash", "train_order_key"]
    ]
    .sort_values("train_order_key", kind="mergesort")
    .reset_index(drop=True)
)
train_order_df["train_rank"] = np.arange(
    1, len(train_order_df) + 1, dtype=np.int64
)

assert set(dev_rows).isdisjoint(set(val_rows))
assert set(dev_rows).isdisjoint(frozen_exclusion_rows)
assert set(val_rows).isdisjoint(frozen_exclusion_rows)

print("\nSplit counts")
display(meta["split"].value_counts().rename("count").to_frame())

split_manifest_path = DIR_MANIFESTS / "imb_split_manifest.parquet"
train_order_path = DIR_MANIFESTS / "imb_train_nested_order.parquet"

meta.to_parquet(split_manifest_path, index=False)
train_order_df.to_parquet(train_order_path, index=False)

print("Saved:", split_manifest_path)
print("Saved:", train_order_path)


Candidate after exclusion/cleaning: 782192

Split counts


,count
split,
train_pool,779192
validation,2000
dev,1000
frozen_test_excluded,1000


Saved: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/manifests/imb_split_manifest.parquet
Saved: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/manifests/imb_train_nested_order.parquet


In [11]:
# ======================================================================================
# CELL 11 — DISTRIBUZIONE PER GENERAL CATEGORY: DEV / VAL / TRAIN
# ======================================================================================

def distribution_table(meta_df: pd.DataFrame) -> pd.DataFrame:
    temp = meta_df.copy()
    temp["General Category"] = (
        temp["General Category"]
        .fillna("__MISSING__")
        .replace("", "__MISSING__")
    )

    keep_splits = ["dev", "validation", "train_pool"]
    rows = []

    full_trainable = temp[temp["split"].isin(keep_splits)]
    base_counts = full_trainable["General Category"].value_counts()
    base_pct = base_counts / base_counts.sum() * 100

    for split in keep_splits:
        sub = temp[temp["split"] == split]
        c = sub["General Category"].value_counts()
        pct = c / len(sub) * 100

        for cat in sorted(set(base_counts.index) | set(c.index)):
            rows.append({
                "split": split,
                "General Category": cat,
                "count": int(c.get(cat, 0)),
                "percentage": float(pct.get(cat, 0.0)),
                "trainable_reference_percentage": float(base_pct.get(cat, 0.0)),
                "delta_pp": float(pct.get(cat, 0.0) - base_pct.get(cat, 0.0)),
            })

    return pd.DataFrame(rows)

dist_df = distribution_table(meta)
display(dist_df)

dist_path = DIR_REPORTS / "split_general_category_distribution.csv"
dist_df.to_csv(dist_path, index=False)
print("Saved:", dist_path)


,split,General Category,count,percentage,trainable_reference_percentage,delta_pp
0,dev,"Cardiologia, Sistema Circolatorio ed Ematologia",104,10.400000,10.372389,0.027611
1,dev,"Dermatologia, Allergie e Estetica",63,6.300000,6.288482,0.011518
2,dev,Gastroenterologia e Salute Digestiva,133,13.300000,13.340586,-0.040586
3,dev,Ginecologia e Salute Femminile,84,8.400000,8.398449,0.001551
4,dev,Medicina Generale e Chirurgia Generale,112,11.200000,11.210675,-0.010675
5,dev,Neurologia,60,6.000000,5.958128,0.041872
6,dev,"Oculistica, Otorinolaringoiatria, Odontoiatria...",107,10.700000,10.689191,0.010809
7,dev,Ortopedia e Sistema Muscoloscheletrico,64,6.400000,6.415688,-0.015688
8,dev,Salute Mentale,133,13.300000,13.269504,0.030496
9,dev,"Urologia, Andrologia e Salute Maschile",140,14.000000,14.056907,-0.056907


Saved: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/split_general_category_distribution.csv


In [12]:
# ======================================================================================
# CELL 12 — MATERIALIZZAZIONE DEV / VALIDATION / PILOT
# ======================================================================================

def dataset_from_row_indices(source_ds: Dataset, row_indices: Iterable[int]) -> Dataset:
    rows = [int(x) for x in row_indices]
    return source_ds.select(rows)


dev_ds = dataset_from_row_indices(
    full_ds,
    meta.loc[meta["split"] == "dev", "row_index"].astype(int).tolist()
)

validation_ds = dataset_from_row_indices(
    full_ds,
    meta.loc[meta["split"] == "validation", "row_index"].astype(int).tolist()
)

dev_path = DIR_DATASETS / "IMB_DEV_1000"
val_path = DIR_DATASETS / "IMB_VALIDATION_2000"

dev_ds.save_to_disk(str(dev_path))
validation_ds.save_to_disk(str(val_path))

print("DEV saved       :", dev_path, len(dev_ds))
print("VALIDATION saved:", val_path, len(validation_ds))


def get_nested_train_dataset(n: int) -> Dataset:
    """Materializza i primi n record dell'ordine train congelato."""
    if n < 1 or n > len(train_order_df):
        raise ValueError(f"n={n} fuori range 1..{len(train_order_df)}")

    rows = train_order_df.iloc[:n]["row_index"].astype(int).tolist()
    return dataset_from_row_indices(full_ds, rows)


pilot_ds = get_nested_train_dataset(PILOT_SIZE)
pilot_arrow_path = DIR_PILOT / "IMB_PILOT_1000"
pilot_ds.save_to_disk(str(pilot_arrow_path))

print("PILOT saved:", pilot_arrow_path, len(pilot_ds))


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

DEV saved       : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/datasets/IMB_DEV_1000 1000
VALIDATION saved: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/datasets/IMB_VALIDATION_2000 2000


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

PILOT saved: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/pilot/IMB_PILOT_1000 1000


## Fase 2 — Conversione SFT per Unsloth Studio

Formato JSONL:

```json
{
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "<Question originale>"},
    {"role": "assistant", "content": "<Answer originale>"}
  ],
  "metadata": {
    "ID": "...",
    "General Category": "...",
    "Category": "..."
  }
}
```

La metadata serve alla tracciabilità; Question e Answer non vengono riscritte semanticamente.


In [16]:
# ======================================================================================
# CELL 13 — EXPORT SFT JSONL
# ======================================================================================

def to_sft_record(row: dict) -> dict:
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_SFT},
            {
                "role": "user",
                "content": "" if row["Question"] is None else str(row["Question"])
            },
            {
                "role": "assistant",
                "content": "" if row["Answer"] is None else str(row["Answer"])
            },
        ],
        "metadata": {
            "ID": "" if row.get("ID") is None else str(row.get("ID")),
            "General Category": (
                "" if row.get("General Category") is None
                else str(row.get("General Category"))
            ),
            "Category": (
                "" if row.get("Category") is None
                else str(row.get("Category"))
            ),
        },
    }


def export_sft_jsonl(ds: Dataset, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in ds:
            rec = to_sft_record(row)
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return path


pilot_jsonl_path = export_sft_jsonl(
    pilot_ds,
    DIR_PILOT / "imb_pilot_1000_sft_messages.jsonl"
)
dev_jsonl_path = export_sft_jsonl(
    dev_ds,
    DIR_EXPORTS / "imb_dev_1000_sft_messages.jsonl"
)
val_jsonl_path = export_sft_jsonl(
    validation_ds,
    DIR_EXPORTS / "imb_validation_2000_sft_messages.jsonl"
)

print("Pilot JSONL:", pilot_jsonl_path)
print("DEV JSONL  :", dev_jsonl_path)
print("VAL JSONL  :", val_jsonl_path)


def export_learning_curve_subset(n: int) -> Path:
    """Esporta in futuro uno dei subset annidati senza crearli tutti subito."""
    ds = get_nested_train_dataset(n)
    out_path = DIR_EXPORTS / f"imb_ft_{n//1000:03d}k_sft_messages.jsonl"
    return export_sft_jsonl(ds, out_path)

print("\nFuture nested sizes:", LEARNING_CURVE_SIZES)
print("Esempio futuro: export_learning_curve_subset(5000)")


Pilot JSONL: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/pilot/imb_pilot_1000_sft_messages.jsonl
DEV JSONL  : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_dev_1000_sft_messages.jsonl
VAL JSONL  : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_validation_2000_sft_messages.jsonl

Future nested sizes: [5000, 10000, 25000, 50000, 100000]
Esempio futuro: export_learning_curve_subset(5000)


## Analisi lunghezza in token

La `max_seq_length` è critica sulla RTX 5070 8 GB.

Il notebook prima misura la distribuzione delle lunghezze usando il tokenizer di MedGemma 1.5 4B-it. Il default usa un campione deterministico di 20.000 record; impostare `TOKEN_STATS_SAMPLE_SIZE = None` per un audit completo.


In [17]:
# ======================================================================================
# CELL 14 — TOKENIZER MEDGEMMA E TOKEN-LENGTH AUDIT
# ======================================================================================

from transformers import AutoProcessor, AutoTokenizer

print("Loading tokenizer/processor:", MEDGEMMA_MODEL_ID)

tokenizer = None
processor = None

try:
    processor = AutoProcessor.from_pretrained(
        MEDGEMMA_MODEL_ID,
        cache_dir=str(HF_HOME),
    )
    tokenizer = getattr(processor, "tokenizer", None)
except Exception as e:
    print("AutoProcessor warning:", repr(e))

if tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(
        MEDGEMMA_MODEL_ID,
        cache_dir=str(HF_HOME),
    )

print("Tokenizer:", tokenizer.__class__.__name__)


def sft_token_length(question: str, answer: str) -> int:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_SFT},
        {"role": "user", "content": "" if question is None else str(question)},
        {"role": "assistant", "content": "" if answer is None else str(answer)},
    ]

    try:
        token_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )
        return len(token_ids)
    except Exception:
        text = (
            f"<system>\n{SYSTEM_PROMPT_SFT}\n"
            f"<user>\n{messages[1]['content']}\n"
            f"<assistant>\n{messages[2]['content']}"
        )
        return len(tokenizer(text, add_special_tokens=True)["input_ids"])


if TOKEN_STATS_SAMPLE_SIZE is None:
    token_rows = train_order_df["row_index"].astype(int).tolist()
else:
    token_n = min(int(TOKEN_STATS_SAMPLE_SIZE), len(train_order_df))
    token_rows = train_order_df.iloc[:token_n]["row_index"].astype(int).tolist()

token_ds = dataset_from_row_indices(full_ds, token_rows)

lengths = []
t0 = time.time()

for i, row in enumerate(token_ds):
    lengths.append(sft_token_length(row["Question"], row["Answer"]))
    if (i + 1) % 1000 == 0:
        print(f"Tokenized {i+1:,}/{len(token_ds):,}")

lengths = np.asarray(lengths, dtype=np.int32)

percentiles = [50, 75, 90, 95, 97.5, 99, 99.5, 100]
stats = {f"p{p}": float(np.percentile(lengths, p)) for p in percentiles}

print("\nToken-length statistics")
print("n   :", len(lengths))
print("mean:", float(lengths.mean()))
for k, v in stats.items():
    print(f"{k:>5}: {v:.1f}")

candidate_lengths = [512, 768, 1024, 1280, 1536, 2048, 3072, 4096]
truncation_table = pd.DataFrame({
    "max_seq_length": candidate_lengths,
    "records_over_limit": [int((lengths > x).sum()) for x in candidate_lengths],
    "percentage_over_limit": [
        float((lengths > x).mean() * 100) for x in candidate_lengths
    ],
})

display(truncation_table)

pd.DataFrame({"token_length": lengths}).to_parquet(
    DIR_REPORTS / "token_length_sample.parquet",
    index=False
)
truncation_table.to_csv(
    DIR_REPORTS / "candidate_max_seq_length_truncation.csv",
    index=False
)

print(f"Elapsed: {(time.time()-t0)/60:.2f} min")


Loading tokenizer/processor: google/medgemma-1.5-4b-it


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Tokenizer: GemmaTokenizerFast
Tokenized 1,000/20,000
Tokenized 2,000/20,000
Tokenized 3,000/20,000
Tokenized 4,000/20,000
Tokenized 5,000/20,000
Tokenized 6,000/20,000
Tokenized 7,000/20,000
Tokenized 8,000/20,000
Tokenized 9,000/20,000
Tokenized 10,000/20,000
Tokenized 11,000/20,000
Tokenized 12,000/20,000
Tokenized 13,000/20,000
Tokenized 14,000/20,000
Tokenized 15,000/20,000
Tokenized 16,000/20,000
Tokenized 17,000/20,000
Tokenized 18,000/20,000
Tokenized 19,000/20,000
Tokenized 20,000/20,000

Token-length statistics
n   : 20000
mean: 393.8501
  p50: 340.0
  p75: 489.0
  p90: 696.0
  p95: 842.0
p97.5: 923.0
  p99: 1029.0
p99.5: 1149.0
 p100: 2350.0


,max_seq_length,records_over_limit,percentage_over_limit
0,512,4551,22.755
1,768,1468,7.340
2,1024,208,1.040
3,1280,52,0.260
4,1536,17,0.085
5,2048,1,0.005
6,3072,0,0.000
7,4096,0,0.000


Elapsed: 0.22 min


In [18]:
# ======================================================================================
# CELL 15 — PROPOSTA DATA-DRIVEN DELLA MAX_SEQ_LENGTH
# ======================================================================================

def recommend_seq_length(lengths: np.ndarray) -> int:
    """
    Pilot 8 GB: sceglie il più piccolo candidato che copre almeno il p95,
    con tetto iniziale 2048 per evitare un pilot inutilmente costoso.
    """
    p95 = float(np.percentile(lengths, 95))
    candidates = [768, 1024, 1280, 1536, 2048]
    for c in candidates:
        if c >= p95:
            return c
    return 2048


RECOMMENDED_MAX_SEQ_LENGTH = recommend_seq_length(lengths)

print("Recommended pilot max_seq_length:", RECOMMENDED_MAX_SEQ_LENGTH)
print(
    "Estimated sample truncation rate:",
    f"{(lengths > RECOMMENDED_MAX_SEQ_LENGTH).mean()*100:.2f}%"
)
print(
    "\nRaccomandazione solo per il PILOT. "
    "Dopo il pilot verranno valutati VRAM, throughput e OOM."
)


Recommended pilot max_seq_length: 1024
Estimated sample truncation rate: 1.04%

Raccomandazione solo per il PILOT. Dopo il pilot verranno valutati VRAM, throughput e OOM.


## Fase 3 — Pilot QLoRA locale

Il pilot non misura la qualità finale. Misura fattibilità, VRAM, tempo/step, throughput, temperatura, OOM e dimensione checkpoint.

Configurazione iniziale:

| parametro | valore pilot |
|---|---|
| modello | `google/medgemma-1.5-4b-it` |
| quantizzazione | 4-bit |
| metodo | QLoRA |
| vision fine-tuning | NO |
| language fine-tuning | YES |
| attention LoRA | YES |
| MLP LoRA | YES |
| LoRA rank | 16 |
| LoRA alpha | 16 |
| dropout | 0 |
| batch/device | 1 |
| gradient accumulation | 8 |
| epochs | 1 |
| learning rate | 2e-4 |
| weight decay | 0.01 |
| warmup ratio | 0.05 |
| scheduler | linear |
| gradient checkpointing | YES |
| train on responses only | YES |
| precision | BF16 se disponibile |
| seed | 20260831 |
| max sequence length | definita dalla Cell 15 |


In [19]:
# ======================================================================================
# CELL 16 — GPU / BF16 DIAGNOSTICS
# ======================================================================================

import torch

print("=" * 100)
print("LOCAL TRAINING DIAGNOSTICS")
print("=" * 100)
print("Python       :", sys.version.split()[0])
print("Platform     :", platform.platform())
print("Torch        :", torch.__version__)
print("CUDA avail.  :", torch.cuda.is_available())

gpu_info = {}

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_info = {
        "gpu_name": torch.cuda.get_device_name(0),
        "total_vram_gb": round(props.total_memory / 1024**3, 3),
        "bf16_supported": bool(torch.cuda.is_bf16_supported()),
        "cuda_runtime": torch.version.cuda,
    }
    for k, v in gpu_info.items():
        print(f"{k:16}: {v}")
else:
    print("ATTENZIONE: CUDA non disponibile. Non lanciare il pilot.")

if torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
    print("\nBF16 non segnalato come supportato: verificare precisione in Unsloth Studio.")


LOCAL TRAINING DIAGNOSTICS
Python       : 3.11.14
Platform     : Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.35
Torch        : 2.10.0+cu128
CUDA avail.  : True
gpu_name        : NVIDIA GeForce RTX 5070 Laptop GPU
total_vram_gb   : 7.96
bf16_supported  : True
cuda_runtime    : 12.8


In [20]:
# ======================================================================================
# CELL 17 — CONFIG PILOT PER UNSLOTH STUDIO
# ======================================================================================

pilot_config = {
    "study": {
        "name": FT_STUDY_NAME,
        "version": STUDY_VERSION,
        "baseline": BASELINE_VERSION,
        "seed": FT_SPLIT_SEED,
    },
    "model": {
        "model_id": MEDGEMMA_MODEL_ID,
        "quantization": "4bit",
        "method": "QLoRA",
        "fine_tune_vision": False,
        "fine_tune_language": True,
        "lora_attention": True,
        "lora_mlp": True,
    },
    "dataset": {
        "pilot_jsonl": str(pilot_jsonl_path),
        "pilot_rows": len(pilot_ds),
        "format": "messages",
        "train_on_responses_only": True,
    },
    "training": {
        "epochs": 1,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 8,
        "effective_batch_size": 8,
        "learning_rate": 2e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "linear",
        "lora_rank": 16,
        "lora_alpha": 16,
        "lora_dropout": 0.0,
        "gradient_checkpointing": True,
        "precision_preference": "bf16",
        "max_seq_length": int(RECOMMENDED_MAX_SEQ_LENGTH),
        "seed": FT_SPLIT_SEED,
    },
    "diagnostics_to_record": [
        "training_time_sec",
        "steps",
        "samples_per_sec",
        "tokens_per_sec_if_available",
        "peak_vram_mb",
        "gpu_temperature_max_c",
        "oom_events",
        "checkpoint_size_mb",
        "training_loss",
    ],
}

pilot_config_path = DIR_PILOT / "unsloth_studio_pilot_config.json"
pilot_config_path.write_text(
    json.dumps(pilot_config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(json.dumps(pilot_config, ensure_ascii=False, indent=2))
print("\nSaved:", pilot_config_path)


{
  "study": {
    "name": "EMCRAG-FT",
    "version": "v07.01",
    "baseline": "CRAG_MedGemma_v.06.11",
    "seed": 20260831
  },
  "model": {
    "model_id": "google/medgemma-1.5-4b-it",
    "quantization": "4bit",
    "method": "QLoRA",
    "fine_tune_vision": false,
    "fine_tune_language": true,
    "lora_attention": true,
    "lora_mlp": true
  },
  "dataset": {
    "pilot_jsonl": "/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/pilot/imb_pilot_1000_sft_messages.jsonl",
    "pilot_rows": 1000,
    "format": "messages",
    "train_on_responses_only": true
  },
  "training": {
    "epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "effective_batch_size": 8,
    "learning_rate": 0.0002,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "linear",
    "lora_rank": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,
    "gradient_checkpointing": true,
    "precision_preference": "bf16",
    "max_seq_length": 10

In [21]:
# ======================================================================================
# CELL 18 — FINAL INTEGRITY CHECK + STUDY MANIFEST + FILE HASHES
# ======================================================================================

frozen_hashes = set(frozen_df["_qa_hash"].astype(str))

for split_name in ["dev", "validation", "train_pool"]:
    split_hashes = set(
        meta.loc[meta["split"] == split_name, "qa_hash"]
        .dropna().astype(str)
    )
    overlap = frozen_hashes & split_hashes
    print(f"{split_name:12} frozen-hash overlap:", len(overlap))
    if overlap:
        raise RuntimeError(
            f"DATA LEAKAGE: {len(overlap)} frozen hashes in {split_name}"
        )

sets = {
    s: set(meta.loc[meta["split"] == s, "row_index"].astype(int))
    for s in ["dev", "validation", "train_pool"]
}
assert sets["dev"].isdisjoint(sets["validation"])
assert sets["dev"].isdisjoint(sets["train_pool"])
assert sets["validation"].isdisjoint(sets["train_pool"])

check_sizes = [PILOT_SIZE] + LEARNING_CURVE_SIZES
previous = set()

for n in check_sizes:
    current = set(train_order_df.iloc[:n]["row_index"].astype(int))
    if previous and not previous.issubset(current):
        raise RuntimeError(f"Nested invariant failed at n={n}")
    previous = current

print("Nested subset invariant: OK")

artifact_files = [
    full_meta_path,
    leakage_audit_path,
    split_manifest_path,
    train_order_path,
    pilot_jsonl_path,
    dev_jsonl_path,
    val_jsonl_path,
    pilot_config_path,
]

artifact_hashes = {}
for p in artifact_files:
    if p.exists() and p.is_file():
        artifact_hashes[str(p)] = sha256_file(p)

manifest = {
    "study_name": FT_STUDY_NAME,
    "study_version": STUDY_VERSION,
    "baseline_version": BASELINE_VERSION,
    "created_local_time": time.strftime("%Y-%m-%d %H:%M:%S"),
    "paths": {
        "imb_full": str(IMB_FULL_PATH),
        "imb_frozen_test": str(IMB_FROZEN_TEST_PATH),
        "ft_root": str(FT_ROOT),
    },
    "dataset_observed": {
        "full_rows": len(full_ds),
        "full_columns": full_ds.column_names,
        "frozen_rows": len(frozen_ds),
        "frozen_question_column": FROZEN_Q_COL,
        "frozen_answer_column": FROZEN_A_COL,
        "frozen_id_column": FROZEN_ID_COL,
        "readme_expected_rows": README_EXPECTED_ROWS,
        "attached_plot_total": PLOT_CATEGORY_TOTAL,
    },
    "leakage_audit": {
        "frozen_logical_records": len(frozen_ds),
        "frozen_full_rows_excluded": len(frozen_exclusion_rows),
        "unresolved": int((~leakage_audit_df["resolved"]).sum()),
        "method_a_id_available": bool(METHOD_A_AVAILABLE),
        "hash_matched_frozen_records": int(
            (leakage_audit_df["hash_match_count"] > 0).sum()
        ),
    },
    "splits": {
        "seed": FT_SPLIT_SEED,
        "dev_size": int((meta["split"] == "dev").sum()),
        "validation_size": int((meta["split"] == "validation").sum()),
        "train_pool_size": int((meta["split"] == "train_pool").sum()),
        "invalid_qa": int((meta["split"] == "invalid_qa").sum()),
        "duplicate_removed": int((meta["split"] == "duplicate_removed").sum()),
        "frozen_test_excluded_full_rows": int(
            (meta["split"] == "frozen_test_excluded").sum()
        ),
        "nested_sizes": [PILOT_SIZE] + LEARNING_CURVE_SIZES,
    },
    "sft": {
        "system_prompt": SYSTEM_PROMPT_SFT,
        "format": "messages",
        "question_semantic_rewrite": False,
        "answer_semantic_rewrite": False,
    },
    "token_audit": {
        "sample_size": len(lengths),
        "recommended_pilot_max_seq_length": int(RECOMMENDED_MAX_SEQ_LENGTH),
        "percentiles": stats,
    },
    "gpu": gpu_info,
    "artifact_sha256": artifact_hashes,
}

manifest_path = DIR_MANIFESTS / "EMCRAG_FT_v0701_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 100)
print("PHASES 0–3 PREPARATION COMPLETE")
print("=" * 100)
print("Manifest:", manifest_path)
print("Pilot   :", pilot_jsonl_path)
print("Config  :", pilot_config_path)
print("\nNON eseguire ancora RAGAS/GPT-4o.")
print("Prossimo passo: importare PILOT-1K in Unsloth Studio e lanciare 1 epoch diagnostica.")


dev          frozen-hash overlap: 0
validation   frozen-hash overlap: 0
train_pool   frozen-hash overlap: 0
Nested subset invariant: OK

PHASES 0–3 PREPARATION COMPLETE
Manifest: /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/manifests/EMCRAG_FT_v0701_manifest.json
Pilot   : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/pilot/imb_pilot_1000_sft_messages.jsonl
Config  : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/pilot/unsloth_studio_pilot_config.json

NON eseguire ancora RAGAS/GPT-4o.
Prossimo passo: importare PILOT-1K in Unsloth Studio e lanciare 1 epoch diagnostica.


# Risultato atteso prima di aprire Unsloth Studio

Le celle devono completarsi senza eccezioni e mostrare:

- `Frozen rows = 1000`;
- `Unresolved = 0`;
- `dev frozen-hash overlap = 0`;
- `validation frozen-hash overlap = 0`;
- `train_pool frozen-hash overlap = 0`;
- `Nested subset invariant = OK`;
- una `recommended_pilot_max_seq_length`;
- `imb_pilot_1000_sft_messages.jsonl`;
- `unsloth_studio_pilot_config.json`;
- `EMCRAG_FT_v0701_manifest.json`.

## Stop condition

Se la Cell 7 segnala anche **un solo frozen record non risolto**, non proseguire con split o fine-tuning.

## Dopo il pilot

Registrare almeno:
- tempo totale;
- step totali;
- training loss;
- peak VRAM;
- temperatura massima GPU;
- OOM sì/no;
- dimensione adapter;
- throughput.

Questi dati serviranno a stimare tempi realistici per 5k, 10k, 25k, 50k e 100k.


In [ ]:
# ======================================================================================
# CELL 18 bis — Generate the true Fine Tuning -5K IMB-QA dataset
# ======================================================================================

ft5k_path = export_learning_curve_subset(5000)
print(ft5k_path)

/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_005k_sft_messages.jsonl


In [23]:
# ======================================================================================
# CELL 19 — NESTED STRATIFIED LEARNING-CURVE SUBSETS
# ======================================================================================

STRATIFIED_SIZES = [5_000, 10_000, 25_000, 50_000, 100_000]

# Distribuzione di riferimento: SOLO train_pool
train_meta = meta[meta["split"] == "train_pool"].copy()

train_meta["General Category"] = (
    train_meta["General Category"]
    .fillna("__MISSING__")
    .replace("", "__MISSING__")
)

reference_counts = train_meta["General Category"].value_counts()

# Usa l'ordine deterministico già congelato all'interno di ogni categoria
ordered = train_order_df.copy()
ordered["General Category"] = (
    ordered["General Category"]
    .fillna("__MISSING__")
    .replace("", "__MISSING__")
)

category_ordered_rows = {}

for cat in reference_counts.index:
    category_ordered_rows[cat] = (
        ordered[ordered["General Category"] == cat]
        .sort_values("train_order_key", kind="mergesort")["row_index"]
        .astype(int)
        .tolist()
    )

previous_rows = set()
stratified_summary = []

for n in STRATIFIED_SIZES:

    quotas = allocate_quotas(reference_counts, n)

    selected_rows = []

    for cat, quota in quotas.items():
        selected_rows.extend(category_ordered_rows[cat][:quota])

    selected_rows = set(selected_rows)

    # Verifica nested property
    if previous_rows and not previous_rows.issubset(selected_rows):
        raise RuntimeError(
            f"Nested stratification failed at n={n}"
        )

    # Ripristina ordine pseudocasuale globale deterministico
    selected_df = (
        train_order_df[
            train_order_df["row_index"].isin(selected_rows)
        ]
        .sort_values("train_order_key", kind="mergesort")
    )

    final_rows = selected_df["row_index"].astype(int).tolist()

    assert len(final_rows) == n
    assert len(set(final_rows)) == n

    ds = dataset_from_row_indices(full_ds, final_rows)

    out_path = (
        DIR_EXPORTS /
        f"imb_ft_{n//1000:03d}k_STRAT_sft_messages.jsonl"
    )

    export_sft_jsonl(ds, out_path)

    # Distribution audit
    counts = (
        selected_df["General Category"]
        .value_counts()
    )

    for cat in reference_counts.index:
        sample_pct = counts.get(cat, 0) / n * 100
        ref_pct = reference_counts[cat] / reference_counts.sum() * 100

        stratified_summary.append({
            "size": n,
            "General Category": cat,
            "count": int(counts.get(cat, 0)),
            "percentage": sample_pct,
            "reference_percentage": ref_pct,
            "delta_pp": sample_pct - ref_pct,
        })

    print(
        f"{n:>6,} records -> {out_path}"
    )

    previous_rows = selected_rows


stratified_summary_df = pd.DataFrame(stratified_summary)

display(stratified_summary_df)

stratified_summary_df.to_csv(
    DIR_REPORTS / "learning_curve_stratified_distribution.csv",
    index=False
)

print("\nNested stratified learning curve: OK")

 5,000 records -> /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_005k_STRAT_sft_messages.jsonl
10,000 records -> /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_010k_STRAT_sft_messages.jsonl
25,000 records -> /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_025k_STRAT_sft_messages.jsonl
50,000 records -> /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_050k_STRAT_sft_messages.jsonl
100,000 records -> /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_100k_STRAT_sft_messages.jsonl


,size,General Category,count,percentage,reference_percentage,delta_pp
0,5000,"Urologia, Andrologia e Salute Maschile",703,14.060,14.056998,0.003002
1,5000,Gastroenterologia e Salute Digestiva,667,13.340,13.340614,-0.000614
2,5000,Salute Mentale,663,13.260,13.269515,-0.009515
3,5000,Medicina Generale e Chirurgia Generale,561,11.220,11.210716,0.009284
4,5000,"Oculistica, Otorinolaringoiatria, Odontoiatria...",534,10.680,10.689150,-0.009150
5,5000,"Cardiologia, Sistema Circolatorio ed Ematologia",519,10.380,10.372283,0.007717
6,5000,Ginecologia e Salute Femminile,420,8.400,8.398444,0.001556
7,5000,Ortopedia e Sistema Muscoloscheletrico,321,6.420,6.415749,0.004251
8,5000,"Dermatologia, Allergie e Estetica",314,6.280,6.288437,-0.008437
9,5000,Neurologia,298,5.960,5.958095,0.001905



Nested stratified learning curve: OK


In [ ]:
# ======================================================================================
# CELL 20 — STANDALONE AUDIT: TRUNCATION @1024 + ASSISTANT RESPONSE SURVIVAL
# ======================================================================================
#
# EMCRAG Fine-Tuning Experimental Protocol v07.01
#
# IMPORTANT — THIS CELL IS INTENTIONALLY STANDALONE / AUTONOMOUS
# --------------------------------------------------------------
#
# This cell has been deliberately designed so that it can be executed BY ITSELF,
# even after:
#
#   - restarting the Jupyter kernel;
#   - closing and reopening VS Code / the notebook;
#   - restarting WSL2;
#   - restarting the computer;
#   - running the notebook in a new Python session.
#
# CELLS 1–19 DO NOT NEED TO BE EXECUTED BEFORE THIS CELL.
#
# All datasets needed by this audit have already been materialized and saved
# persistently on disk by the previous preparation phases. This cell therefore
# reconstructs everything it needs locally:
#
#   1. imports its own Python dependencies;
#   2. defines the required project paths;
#   3. reads the already exported TRAIN-5K and VALIDATION-2K JSONL files;
#   4. loads the MedGemma tokenizer directly from the configured model;
#   5. applies the MedGemma chat template;
#   6. independently performs the truncation / assistant-mask audit;
#   7. writes persistent audit reports to disk.
#
# Consequently, DO NOT rerun Cells 1–19 merely in order to run Cell 20.
# Re-running them would be unnecessary for this audit and could regenerate
# intermediate artifacts that are already frozen and available on disk.
#
# This cell is READ-ONLY with respect to the training and validation datasets:
# it does not modify, shuffle, resample or overwrite their contents.
#
# It can therefore also be SAFELY RE-RUN later, as many times as required,
# for reproducibility, verification, debugging, or documentation of the
# experiment. A later execution should produce the same results as long as:
#
#   - the input JSONL files remain unchanged;
#   - the MedGemma tokenizer/chat template remains unchanged;
#   - MAX_SEQ_LENGTH remains 1024.
#
# INPUT DATASETS
# --------------
#
# TRAIN:
#   /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/
#   imb_ft_005k_STRAT_sft_messages.jsonl
#
# VALIDATION:
#   /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/
#   imb_validation_2000_sft_messages.jsonl
#
# WHY THIS AUDIT IS BEING PERFORMED
# ---------------------------------
#
# The FT-5K Unsloth Studio run was configured with:
#
#   source TRAIN records      = 5,000
#   source VALIDATION records = 2,000
#   max_seq_length            = 1024
#   physical batch size       = 1
#   gradient accumulation     = 8
#   assistant-only loss       = ON
#
# However, the completed Unsloth run showed:
#
#   optimizer steps           = 623
#   evaluation samples/batches ≈ 1,989
#
# These values suggest that the effective number of examples processed by
# Unsloth may differ slightly from the number of rows in the source JSONL.
#
# A working hypothesis is that some long conversations may interact with:
#
#   - MedGemma chat templating;
#   - right truncation at 1024 tokens;
#   - assistant-only loss masking;
#
# in such a way that part or all of the assistant response is removed.
#
# THIS IS ONLY A HYPOTHESIS AT THIS STAGE.
# Cell 20 is intended to test it rather than assume it is true.
#
# AUDIT CLASSIFICATION
# --------------------
#
# Each record is classified as:
#
#   FIT
#       Full system + user + assistant sequence fits within 1024 tokens.
#
#   ASSISTANT_PARTIALLY_TRUNCATED
#       The sequence exceeds 1024 tokens but at least part of the assistant
#       response survives right truncation.
#
#   ASSISTANT_COMPLETELY_TRUNCATED
#       The assistant response begins at or beyond the 1024-token boundary,
#       so no assistant response token would survive right truncation.
#
# Invalid/empty message structures are also detected explicitly.
#
# MAIN QUESTION
# -------------
#
# Determine whether assistant-response truncation can explain the apparent
# reduction observed in Unsloth Studio, provisionally estimated as:
#
#   TRAIN      : 5000 -> approximately 4984
#   VALIDATION : 2000 -> 1989
#
# No conclusion should be drawn until the actual Cell 20 results are examined.
#
# OUTPUT
# ------
#
# The cell prints:
#
#   - total records;
#   - sequences <=1024;
#   - sequences >1024;
#   - partially truncated assistant responses;
#   - completely truncated assistant responses;
#   - sequence-length percentiles;
#   - IDs and metadata of affected records.
#
# It also saves persistent reports in:
#
#   /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/
#
# IMPORTANT EXPERIMENTAL SAFETY NOTES
# -----------------------------------
#
# This cell:
#
#   - DOES NOT train MedGemma;
#   - DOES NOT modify LoRA weights;
#   - DOES NOT invoke Unsloth training;
#   - DOES NOT call OpenAI APIs;
#   - DOES NOT run RAGAS;
#   - DOES NOT use the frozen benchmark/test set;
#   - DOES NOT alter TRAIN, VALIDATION or DEV datasets.
#
# Therefore it is safe to execute independently and safe to rerun later.
#
# NEXT EXPERIMENTAL DECISION
# --------------------------
#
# FT-10K must NOT be started until the results of this audit have been reviewed
# and the effective-record-count discrepancy has either been explained or
# explicitly documented in Experimental Protocol v07.01.
#
# ======================================================================================
#
# RELATION TO THE NEXT EXPERIMENTAL STAGE — CELL 23
# -------------------------------------------------
#
# Cell 20 is a PREPROCESSING AUDIT, not a model-quality evaluation.
# Its purpose is to establish whether max_seq_length=1024 and
# assistant-only loss produce a controlled and acceptable loss of
# trainable examples before continuing the learning-curve experiment.
#
# After Cell 20, its persistent summary is inspected by Cell 21 and the
# result is independently verified using the exact FT-5K checkpoint
# tokenizer/chat template in Cell 22.
#
# Once Cells 20–22 establish that the preprocessing configuration is
# acceptable, the next stage is Cell 23:
#
#     DEV SANITY CHECK — M0 BASE vs M1 FT-5K
#
# Cell 23 has a different purpose. It does NOT investigate token truncation.
# Instead, it performs a small deterministic qualitative comparison on the
# DEV split to verify that FT-5K has not introduced obvious generation
# degeneration before additional GPU time is invested in FT-10K.
#
# The DEV sanity check is NOT a final benchmark and does NOT use the frozen
# test set, RAGAS, OpenAI APIs, retrieval or web search.
#
# Therefore the experimental sequence is:
#
#     Cell 20 -> preprocessing audit
#     Cell 21 -> persistent audit summary
#     Cell 22 -> exact checkpoint-template verification
#     Cell 23 -> DEV qualitative sanity check M0 vs M1
#     FT-10K  -> only if the previous checks pass
#
# Cells 20, 21, 22 and 23 are deliberately standalone and can each be
# executed after a fresh kernel restart without rerunning Cells 1–19.


from pathlib import Path
import json
import pandas as pd
from transformers import AutoTokenizer

# --------------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------------

BASE_DIR = Path("/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01")

TRAIN_JSONL = (
    BASE_DIR
    / "exports"
    / "imb_ft_005k_STRAT_sft_messages.jsonl"
)

VALID_JSONL = (
    BASE_DIR
    / "exports"
    / "imb_validation_2000_sft_messages.jsonl"
)

REPORT_DIR = BASE_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "google/medgemma-1.5-4b-it"
MAX_SEQ_LENGTH = 1024

EXPECTED_TRAIN_SOURCE = 5000
EXPECTED_VALID_SOURCE = 2000

# Counts apparently used by Unsloth Studio
UNSLOTH_TRAIN_EFFECTIVE = 4984
UNSLOTH_VALID_EFFECTIVE = 1989

print("=" * 100)
print("CELL 20 — TRUNCATION / ASSISTANT-MASK AUDIT")
print("=" * 100)

# --------------------------------------------------------------------------------------
# CHECK INPUT FILES
# --------------------------------------------------------------------------------------

for p in (TRAIN_JSONL, VALID_JSONL):
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

print("TRAIN :", TRAIN_JSONL)
print("VALID :", VALID_JSONL)
print()

# --------------------------------------------------------------------------------------
# TOKENIZER
# --------------------------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
)

print("Tokenizer :", tokenizer.__class__.__name__)
print("Model ID  :", MODEL_ID)
print("Max length:", MAX_SEQ_LENGTH)
print()

if tokenizer.chat_template is None:
    raise RuntimeError(
        "Tokenizer has no chat_template. "
        "Audit cannot reproduce the MedGemma conversation format safely."
    )

# --------------------------------------------------------------------------------------
# LOAD JSONL
# --------------------------------------------------------------------------------------

def load_jsonl(path):
    records = []

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception as exc:
                raise RuntimeError(
                    f"Invalid JSON at {path}, line {line_no}"
                ) from exc

            records.append(obj)

    return records


train_records = load_jsonl(TRAIN_JSONL)
valid_records = load_jsonl(VALID_JSONL)

print("Loaded TRAIN records :", len(train_records))
print("Loaded VALID records :", len(valid_records))
print()

if len(train_records) != EXPECTED_TRAIN_SOURCE:
    print(
        f"WARNING: expected {EXPECTED_TRAIN_SOURCE} TRAIN rows, "
        f"found {len(train_records)}"
    )

if len(valid_records) != EXPECTED_VALID_SOURCE:
    print(
        f"WARNING: expected {EXPECTED_VALID_SOURCE} VALID rows, "
        f"found {len(valid_records)}"
    )

# --------------------------------------------------------------------------------------
# AUDIT ONE RECORD
# --------------------------------------------------------------------------------------

def audit_record(record, split_name, row_number):

    messages = record.get("messages", [])
    metadata = record.get("metadata", {}) or {}

    result = {
        "split": split_name,
        "row_number": row_number,
        "ID": metadata.get("ID"),
        "General Category": metadata.get("General Category"),
        "Category": metadata.get("Category"),
        "valid_structure": False,
        "full_tokens": None,
        "prompt_tokens": None,
        "assistant_region_tokens": None,
        "tokens_over_1024": None,
        "assistant_tokens_surviving_1024": None,
        "status": None,
    }

    # Expected:
    # system -> user -> assistant
    if (
        len(messages) != 3
        or messages[0].get("role") != "system"
        or messages[1].get("role") != "user"
        or messages[2].get("role") != "assistant"
    ):
        result["status"] = "INVALID_STRUCTURE"
        return result

    system_text = str(messages[0].get("content", "") or "")
    user_text = str(messages[1].get("content", "") or "")
    assistant_text = str(messages[2].get("content", "") or "")

    if not user_text.strip():
        result["status"] = "EMPTY_USER"
        return result

    if not assistant_text.strip():
        result["status"] = "EMPTY_ASSISTANT"
        return result

    result["valid_structure"] = True

    # ------------------------------------------------------------------
    # Full training conversation:
    # system + user + assistant
    # ------------------------------------------------------------------

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
    )

    # ------------------------------------------------------------------
    # Prompt immediately before assistant answer:
    # system + user + assistant-generation header
    #
    # The first answer token must occur after this prefix.
    # ------------------------------------------------------------------

    prompt_messages = messages[:2]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    full_len = len(full_ids)
    prompt_len = len(prompt_ids)

    assistant_region = max(0, full_len - prompt_len)

    # Number of assistant-region tokens that would remain after
    # right truncation to MAX_SEQ_LENGTH.
    surviving = max(
        0,
        min(full_len, MAX_SEQ_LENGTH) - prompt_len
    )

    result["full_tokens"] = full_len
    result["prompt_tokens"] = prompt_len
    result["assistant_region_tokens"] = assistant_region
    result["tokens_over_1024"] = max(0, full_len - MAX_SEQ_LENGTH)
    result["assistant_tokens_surviving_1024"] = surviving

    # ------------------------------------------------------------------
    # Classification
    # ------------------------------------------------------------------

    if full_len <= MAX_SEQ_LENGTH:
        status = "FIT"

    elif prompt_len >= MAX_SEQ_LENGTH:
        # Right truncation ends before assistant content begins.
        status = "ASSISTANT_COMPLETELY_TRUNCATED"

    else:
        # Some assistant tokens survive but response is incomplete.
        status = "ASSISTANT_PARTIALLY_TRUNCATED"

    result["status"] = status

    return result


# --------------------------------------------------------------------------------------
# RUN AUDIT
# --------------------------------------------------------------------------------------

def audit_dataset(records, split_name):

    rows = []

    for i, record in enumerate(records):
        rows.append(
            audit_record(
                record=record,
                split_name=split_name,
                row_number=i,
            )
        )

    return pd.DataFrame(rows)


train_audit = audit_dataset(
    train_records,
    "TRAIN_5K"
)

valid_audit = audit_dataset(
    valid_records,
    "VALIDATION_2K"
)

audit_df = pd.concat(
    [train_audit, valid_audit],
    ignore_index=True,
)

# --------------------------------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------------------------------

def print_summary(df, split_name, unsloth_effective=None):

    x = df[df["split"] == split_name].copy()

    total = len(x)

    fit = (x["status"] == "FIT").sum()

    partial = (
        x["status"]
        == "ASSISTANT_PARTIALLY_TRUNCATED"
    ).sum()

    complete = (
        x["status"]
        == "ASSISTANT_COMPLETELY_TRUNCATED"
    ).sum()

    invalid = (
        ~x["status"].isin(
            [
                "FIT",
                "ASSISTANT_PARTIALLY_TRUNCATED",
                "ASSISTANT_COMPLETELY_TRUNCATED",
            ]
        )
    ).sum()

    over_1024 = (
        pd.to_numeric(
            x["full_tokens"],
            errors="coerce"
        ) > MAX_SEQ_LENGTH
    ).sum()

    print("-" * 100)
    print(split_name)
    print("-" * 100)

    print(f"Total records                         : {total}")
    print(f"Full sequence <=1024                  : {fit}")
    print(f"Full sequence >1024                   : {over_1024}")
    print(f"Assistant partially truncated         : {partial}")
    print(f"Assistant completely truncated        : {complete}")
    print(f"Invalid/empty structure               : {invalid}")

    if unsloth_effective is not None:

        observed_removed = total - unsloth_effective

        print()
        print(f"Unsloth apparent effective records    : {unsloth_effective}")
        print(f"Unsloth apparent removed records      : {observed_removed}")
        print(
            "Complete assistant truncation count   : "
            f"{complete}"
        )

        if complete == observed_removed:
            print(
                "MATCH: complete assistant truncation "
                "exactly explains the Unsloth reduction."
            )
        else:
            print(
                "NO EXACT MATCH: truncation alone does NOT "
                "fully explain the Unsloth reduction."
            )

    print()


print()
print("=" * 100)
print("AUDIT SUMMARY")
print("=" * 100)

print_summary(
    audit_df,
    "TRAIN_5K",
    UNSLOTH_TRAIN_EFFECTIVE,
)

print_summary(
    audit_df,
    "VALIDATION_2K",
    UNSLOTH_VALID_EFFECTIVE,
)

# --------------------------------------------------------------------------------------
# DISTRIBUTION OF SEQUENCE LENGTHS
# --------------------------------------------------------------------------------------

print("=" * 100)
print("SEQUENCE LENGTH DISTRIBUTION")
print("=" * 100)

for split in ["TRAIN_5K", "VALIDATION_2K"]:

    x = pd.to_numeric(
        audit_df.loc[
            audit_df["split"] == split,
            "full_tokens"
        ],
        errors="coerce",
    ).dropna()

    print(split)

    for q in [0.50, 0.75, 0.90, 0.95, 0.975, 0.99]:

        print(
            f"  p{q*100:>5.1f}: "
            f"{x.quantile(q):.0f}"
        )

    print(
        f"  max   : {x.max():.0f}"
    )

    print()

# --------------------------------------------------------------------------------------
# PROBLEM RECORDS
# --------------------------------------------------------------------------------------

problem_mask = audit_df["status"].isin(
    [
        "ASSISTANT_PARTIALLY_TRUNCATED",
        "ASSISTANT_COMPLETELY_TRUNCATED",
        "INVALID_STRUCTURE",
        "EMPTY_USER",
        "EMPTY_ASSISTANT",
    ]
)

problem_df = (
    audit_df[problem_mask]
    .sort_values(
        ["split", "status", "full_tokens"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

print("=" * 100)
print("PROBLEM / TRUNCATED RECORDS")
print("=" * 100)

display(
    problem_df[
        [
            "split",
            "row_number",
            "ID",
            "General Category",
            "Category",
            "full_tokens",
            "prompt_tokens",
            "assistant_region_tokens",
            "assistant_tokens_surviving_1024",
            "status",
        ]
    ]
)

# --------------------------------------------------------------------------------------
# SAVE REPORTS
# --------------------------------------------------------------------------------------

audit_path = (
    REPORT_DIR
    / "cell20_truncation_assistant_mask_audit.csv"
)

problem_path = (
    REPORT_DIR
    / "cell20_truncation_problem_records.csv"
)

summary_path = (
    REPORT_DIR
    / "cell20_truncation_summary.json"
)

audit_df.to_csv(
    audit_path,
    index=False,
)

problem_df.to_csv(
    problem_path,
    index=False,
)

summary = {}

for split, expected_effective in [
    ("TRAIN_5K", UNSLOTH_TRAIN_EFFECTIVE),
    ("VALIDATION_2K", UNSLOTH_VALID_EFFECTIVE),
]:

    x = audit_df[audit_df["split"] == split]

    summary[split] = {
        "total": int(len(x)),
        "fit": int((x["status"] == "FIT").sum()),
        "assistant_partially_truncated": int(
            (
                x["status"]
                == "ASSISTANT_PARTIALLY_TRUNCATED"
            ).sum()
        ),
        "assistant_completely_truncated": int(
            (
                x["status"]
                == "ASSISTANT_COMPLETELY_TRUNCATED"
            ).sum()
        ),
        "apparent_unsloth_effective": int(
            expected_effective
        ),
        "apparent_unsloth_removed": int(
            len(x) - expected_effective
        ),
    }

with summary_path.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )

print()
print("=" * 100)
print("CELL 20 COMPLETE")
print("=" * 100)

print("Full audit :", audit_path)
print("Problems   :", problem_path)
print("Summary    :", summary_path)

print()
print(
    "NEXT STEP: compare the complete-assistant-truncation "
    "counts with Unsloth's apparent 16 TRAIN / 11 VALIDATION exclusions."
)

CELL 20 — TRUNCATION / ASSISTANT-MASK AUDIT
TRAIN : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_005k_STRAT_sft_messages.jsonl
VALID : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_validation_2000_sft_messages.jsonl

Tokenizer : GemmaTokenizerFast
Model ID  : google/medgemma-1.5-4b-it
Max length: 1024

Loaded TRAIN records : 5000
Loaded VALID records : 2000


AUDIT SUMMARY
----------------------------------------------------------------------------------------------------
TRAIN_5K
----------------------------------------------------------------------------------------------------
Total records                         : 5000
Full sequence <=1024                  : 4946
Full sequence >1024                   : 54
Assistant partially truncated         : 34
Assistant completely truncated        : 20
Invalid/empty structure               : 0

Unsloth apparent effective records    : 4984
Unsloth apparent removed records      : 16
Complete assistant 

,split,row_number,ID,General Category,Category,full_tokens,prompt_tokens,assistant_region_tokens,assistant_tokens_surviving_1024,status
0,TRAIN_5K,3929,524538,"Urologia, Andrologia e Salute Maschile",Urologia,2350,1206,1144,0,ASSISTANT_COMPLETELY_TRUNCATED
1,TRAIN_5K,255,373212,Ortopedia e Sistema Muscoloscheletrico,Ortopedia,1927,1834,93,0,ASSISTANT_COMPLETELY_TRUNCATED
2,TRAIN_5K,268,122890,"Oculistica, Otorinolaringoiatria, Odontoiatria...",Pneumologia,1756,1709,47,0,ASSISTANT_COMPLETELY_TRUNCATED
3,TRAIN_5K,4412,3227,"Dermatologia, Allergie e Estetica",Allergologia e immunologia,1686,1571,115,0,ASSISTANT_COMPLETELY_TRUNCATED
4,TRAIN_5K,1803,154106,"Cardiologia, Sistema Circolatorio ed Ematologia",Ematologia,1624,1506,118,0,ASSISTANT_COMPLETELY_TRUNCATED
...,...,...,...,...,...,...,...,...,...,...
73,VALIDATION_2K,812,346058,"Oculistica, Otorinolaringoiatria, Odontoiatria...",Odontoiatria e odontostomatologia,1067,880,187,144,ASSISTANT_PARTIALLY_TRUNCATED
74,VALIDATION_2K,243,103854,Ortopedia e Sistema Muscoloscheletrico,Medicina fisica e riabilitativa,1047,904,143,120,ASSISTANT_PARTIALLY_TRUNCATED
75,VALIDATION_2K,1032,435557,"Cardiologia, Sistema Circolatorio ed Ematologia",Cardiologia,1044,837,207,187,ASSISTANT_PARTIALLY_TRUNCATED
76,VALIDATION_2K,751,317922,"Oculistica, Otorinolaringoiatria, Odontoiatria...",Oculistica,1033,882,151,142,ASSISTANT_PARTIALLY_TRUNCATED



CELL 20 COMPLETE
Full audit : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell20_truncation_assistant_mask_audit.csv
Problems   : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell20_truncation_problem_records.csv
Summary    : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell20_truncation_summary.json

NEXT STEP: compare the complete-assistant-truncation counts with Unsloth's apparent 16 TRAIN / 11 VALIDATION exclusions.


In [2]:
# ======================================================================================
# CELL 21 — STANDALONE DISPLAY OF CELL 20 AUDIT SUMMARY
# ======================================================================================
#
# This cell is standalone.
# Cells 1–20 do NOT need to be executed.
#
# It only reads the JSON summary already produced by Cell 20.
# It does not modify datasets, models, checkpoints or training artifacts.
# It can be safely rerun at any time.
# ======================================================================================

from pathlib import Path
import json

SUMMARY_PATH = Path(
    "/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/"
    "reports/cell20_truncation_summary.json"
)

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Missing summary file: {SUMMARY_PATH}")

with SUMMARY_PATH.open("r", encoding="utf-8") as f:
    summary = json.load(f)

print("=" * 100)
print("CELL 21 — CELL 20 AUDIT SUMMARY")
print("=" * 100)

for split, x in summary.items():

    print(f"\n{split}")
    print("-" * 100)

    total = x["total"]
    fit = x["fit"]
    partial = x["assistant_partially_truncated"]
    complete = x["assistant_completely_truncated"]
    effective = x["apparent_unsloth_effective"]
    removed = x["apparent_unsloth_removed"]

    print(f"Source records                    : {total}")
    print(f"FIT                               : {fit}")
    print(f"Assistant partially truncated     : {partial}")
    print(f"Assistant completely truncated    : {complete}")
    print(f"Apparent Unsloth effective        : {effective}")
    print(f"Apparent Unsloth removed          : {removed}")

    if complete == removed:
        print("RESULT                            : EXACT MATCH")
    else:
        print(
            f"RESULT                            : NO MATCH "
            f"(difference = {complete - removed:+d})"
        )

print("\n" + "=" * 100)
print("CELL 21 COMPLETE")
print("=" * 100)

CELL 21 — CELL 20 AUDIT SUMMARY

TRAIN_5K
----------------------------------------------------------------------------------------------------
Source records                    : 5000
FIT                               : 4946
Assistant partially truncated     : 34
Assistant completely truncated    : 20
Apparent Unsloth effective        : 4984
Apparent Unsloth removed          : 16
RESULT                            : NO MATCH (difference = +4)

VALIDATION_2K
----------------------------------------------------------------------------------------------------
Source records                    : 2000
FIT                               : 1976
Assistant partially truncated     : 13
Assistant completely truncated    : 11
Apparent Unsloth effective        : 1989
Apparent Unsloth removed          : 11
RESULT                            : EXACT MATCH

CELL 21 COMPLETE


In [3]:
# ======================================================================================
# CELL 22 — STANDALONE EXACT-TRAINING-TEMPLATE AUDIT
# ======================================================================================
#
# EMCRAG Fine-Tuning Experimental Protocol v07.01
#
# PURPOSE
# -------
# Cell 20 used the tokenizer/chat template loaded from:
#
#     google/medgemma-1.5-4b-it
#
# The actual FT-5K training, however, was executed in Unsloth Studio using:
#
#     medgemma-1.5-4b-it-unsloth-bnb-4bit
#
# and the completed run saved its effective tokenizer/chat template inside
# checkpoint-623.
#
# This cell therefore repeats the truncation audit using the EXACT tokenizer
# and chat template saved with the completed FT-5K checkpoint.
#
# IMPORTANT — THIS CELL IS COMPLETELY STANDALONE
# ------------------------------------------------
#
# - Cells 1–21 do NOT need to be executed.
# - It can be run after a fresh kernel restart.
# - It reads only already materialized JSONL/checkpoint files.
# - It does NOT modify datasets or model weights.
# - It does NOT perform training.
# - It does NOT invoke OpenAI or RAGAS.
# - It can safely be rerun later.
#
# MAIN QUESTIONS
# --------------
#
# 1. Does the exact checkpoint tokenizer/template still produce:
#
#       TRAIN:      20 completely truncated assistant responses?
#       VALIDATION: 11 completely truncated assistant responses?
#
# 2. Is the checkpoint chat template identical to the base MedGemma template
#    used in Cell 20?
#
# 3. Is the prompt tokenization truly a prefix of the complete conversation
#    tokenization for every record?
#
# Only after this audit should the preprocessing protocol be frozen for FT-10K.
#
# ======================================================================================

from pathlib import Path
import json
import hashlib
import pandas as pd
from transformers import AutoTokenizer


# --------------------------------------------------------------------------------------
# PATHS
# --------------------------------------------------------------------------------------

BASE_DIR = Path(
    "/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01"
)

TRAIN_JSONL = (
    BASE_DIR
    / "exports"
    / "imb_ft_005k_STRAT_sft_messages.jsonl"
)

VALID_JSONL = (
    BASE_DIR
    / "exports"
    / "imb_validation_2000_sft_messages.jsonl"
)

RUN_DIR = Path(
    "/home/eugrug/.unsloth/studio/outputs/"
    "unsloth_medgemma-1.5-4b-it-unsloth-bnb-4bit"
    "__project-emcrag_ft_imb_005k_v0701_1788215983"
)

CHECKPOINT_DIR = RUN_DIR / "checkpoint-623"

REPORT_DIR = BASE_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 1024


# --------------------------------------------------------------------------------------
# BASIC CHECKS
# --------------------------------------------------------------------------------------

print("=" * 100)
print("CELL 22 — EXACT FT-5K CHECKPOINT TOKENIZER / TEMPLATE AUDIT")
print("=" * 100)

for p in [
    TRAIN_JSONL,
    VALID_JSONL,
    CHECKPOINT_DIR,
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required path: {p}")

print("TRAIN      :", TRAIN_JSONL)
print("VALIDATION :", VALID_JSONL)
print("CHECKPOINT :", CHECKPOINT_DIR)
print()


# --------------------------------------------------------------------------------------
# LOAD EXACT CHECKPOINT TOKENIZER
# --------------------------------------------------------------------------------------

checkpoint_tokenizer = AutoTokenizer.from_pretrained(
    str(CHECKPOINT_DIR),
    use_fast=True,
)

print("Checkpoint tokenizer class:",
      checkpoint_tokenizer.__class__.__name__)

if checkpoint_tokenizer.chat_template is None:
    raise RuntimeError(
        "Checkpoint tokenizer has no chat_template."
    )


# --------------------------------------------------------------------------------------
# LOAD BASE TOKENIZER ONLY FOR TEMPLATE COMPARISON
# --------------------------------------------------------------------------------------

base_tokenizer = AutoTokenizer.from_pretrained(
    "google/medgemma-1.5-4b-it",
    use_fast=True,
)

if base_tokenizer.chat_template is None:
    raise RuntimeError(
        "Base tokenizer has no chat_template."
    )


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


base_template_hash = sha256_text(
    base_tokenizer.chat_template
)

checkpoint_template_hash = sha256_text(
    checkpoint_tokenizer.chat_template
)


print()
print("=" * 100)
print("CHAT TEMPLATE COMPARISON")
print("=" * 100)

print("Base template SHA256      :",
      base_template_hash)

print("Checkpoint template SHA256:",
      checkpoint_template_hash)

print(
    "Templates identical       :",
    base_tokenizer.chat_template
    == checkpoint_tokenizer.chat_template
)

print()


# --------------------------------------------------------------------------------------
# OPTIONAL: EXAMINE UNSLOTH ROW-BOUND METADATA
# --------------------------------------------------------------------------------------

row_bound_path = RUN_DIR / "unsloth_row_bound.json"

print("=" * 100)
print("UNSLOTH ROW-BOUND METADATA")
print("=" * 100)

if row_bound_path.exists():

    try:
        with row_bound_path.open(
            "r",
            encoding="utf-8",
        ) as f:
            row_bound = json.load(f)

        print(
            json.dumps(
                row_bound,
                ensure_ascii=False,
                indent=2,
            )
        )

    except Exception as exc:
        print(
            "Could not parse unsloth_row_bound.json:",
            repr(exc),
        )

else:
    print("unsloth_row_bound.json not found.")

print()


# --------------------------------------------------------------------------------------
# LOAD JSONL
# --------------------------------------------------------------------------------------

def load_jsonl(path):

    records = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        for line_number, line in enumerate(
            f,
            start=1,
        ):

            line = line.strip()

            if not line:
                continue

            try:
                records.append(
                    json.loads(line)
                )

            except Exception as exc:
                raise RuntimeError(
                    f"Invalid JSON: {path}, "
                    f"line {line_number}"
                ) from exc

    return records


train_records = load_jsonl(TRAIN_JSONL)
valid_records = load_jsonl(VALID_JSONL)

print("=" * 100)
print("SOURCE DATASET COUNTS")
print("=" * 100)

print("TRAIN      :", len(train_records))
print("VALIDATION :", len(valid_records))
print()


# --------------------------------------------------------------------------------------
# LONGEST COMMON PREFIX
# --------------------------------------------------------------------------------------

def longest_common_prefix_length(a, b):

    n = min(len(a), len(b))

    i = 0

    while i < n and a[i] == b[i]:
        i += 1

    return i


# --------------------------------------------------------------------------------------
# AUDIT ONE RECORD USING EXACT CHECKPOINT TOKENIZER
# --------------------------------------------------------------------------------------

def audit_record(
    record,
    split_name,
    row_number,
):

    messages = record.get(
        "messages",
        [],
    )

    metadata = (
        record.get("metadata", {})
        or {}
    )

    out = {
        "split": split_name,
        "row_number": row_number,
        "ID": metadata.get("ID"),
        "General Category":
            metadata.get("General Category"),
        "Category":
            metadata.get("Category"),
        "full_tokens": None,
        "prompt_tokens": None,
        "lcp_tokens": None,
        "prompt_is_exact_prefix": None,
        "assistant_region_tokens": None,
        "assistant_tokens_surviving_1024": None,
        "status": None,
    }

    if (
        len(messages) != 3
        or messages[0].get("role") != "system"
        or messages[1].get("role") != "user"
        or messages[2].get("role") != "assistant"
    ):

        out["status"] = "INVALID_STRUCTURE"
        return out

    if not str(
        messages[1].get("content", "")
    ).strip():

        out["status"] = "EMPTY_USER"
        return out

    if not str(
        messages[2].get("content", "")
    ).strip():

        out["status"] = "EMPTY_ASSISTANT"
        return out


    # Complete system-user-assistant conversation
    full_ids = checkpoint_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
    )


    # Exact prefix immediately before assistant response
    prompt_ids = checkpoint_tokenizer.apply_chat_template(
        messages[:2],
        tokenize=True,
        add_generation_prompt=True,
    )


    full_len = len(full_ids)
    prompt_len = len(prompt_ids)

    lcp_len = longest_common_prefix_length(
        full_ids,
        prompt_ids,
    )

    exact_prefix = (
        full_ids[:prompt_len]
        == prompt_ids
    )


    # If prompt_ids is an exact prefix, prompt_len is the
    # appropriate assistant boundary.
    #
    # If not, retain the diagnostic rather than silently
    # assuming that Cell 20's boundary logic is exact.

    assistant_boundary = prompt_len

    assistant_region = max(
        0,
        full_len - assistant_boundary,
    )

    surviving = max(
        0,
        min(
            full_len,
            MAX_SEQ_LENGTH,
        )
        - assistant_boundary,
    )


    out["full_tokens"] = full_len
    out["prompt_tokens"] = prompt_len
    out["lcp_tokens"] = lcp_len
    out["prompt_is_exact_prefix"] = exact_prefix

    out["assistant_region_tokens"] = (
        assistant_region
    )

    out[
        "assistant_tokens_surviving_1024"
    ] = surviving


    if full_len <= MAX_SEQ_LENGTH:

        status = "FIT"

    elif assistant_boundary >= MAX_SEQ_LENGTH:

        status = (
            "ASSISTANT_COMPLETELY_TRUNCATED"
        )

    else:

        status = (
            "ASSISTANT_PARTIALLY_TRUNCATED"
        )


    out["status"] = status

    return out


# --------------------------------------------------------------------------------------
# RUN
# --------------------------------------------------------------------------------------

def audit_dataset(
    records,
    split_name,
):

    return pd.DataFrame(
        [
            audit_record(
                record,
                split_name,
                i,
            )
            for i, record
            in enumerate(records)
        ]
    )


train_df = audit_dataset(
    train_records,
    "TRAIN_5K",
)

valid_df = audit_dataset(
    valid_records,
    "VALIDATION_2K",
)

audit_df = pd.concat(
    [
        train_df,
        valid_df,
    ],
    ignore_index=True,
)


# --------------------------------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------------------------------

print()
print("=" * 100)
print("EXACT CHECKPOINT TOKENIZER AUDIT SUMMARY")
print("=" * 100)


summary = {}

for split in [
    "TRAIN_5K",
    "VALIDATION_2K",
]:

    x = audit_df[
        audit_df["split"] == split
    ]

    total = len(x)

    fit = (
        x["status"] == "FIT"
    ).sum()

    partial = (
        x["status"]
        == "ASSISTANT_PARTIALLY_TRUNCATED"
    ).sum()

    complete = (
        x["status"]
        == "ASSISTANT_COMPLETELY_TRUNCATED"
    ).sum()

    prefix_failures = (
        x["prompt_is_exact_prefix"] == False
    ).sum()

    print()
    print(split)
    print("-" * 100)

    print(
        f"Source records                    : "
        f"{total}"
    )

    print(
        f"FIT                               : "
        f"{fit}"
    )

    print(
        f"Assistant partially truncated     : "
        f"{partial}"
    )

    print(
        f"Assistant completely truncated    : "
        f"{complete}"
    )

    print(
        f"Prompt not exact prefix           : "
        f"{prefix_failures}"
    )

    summary[split] = {
        "source_records":
            int(total),
        "fit":
            int(fit),
        "assistant_partially_truncated":
            int(partial),
        "assistant_completely_truncated":
            int(complete),
        "prompt_not_exact_prefix":
            int(prefix_failures),
    }


# --------------------------------------------------------------------------------------
# SHOW ALL COMPLETELY TRUNCATED RECORDS
# --------------------------------------------------------------------------------------

complete_df = (
    audit_df[
        audit_df["status"]
        == "ASSISTANT_COMPLETELY_TRUNCATED"
    ]
    .sort_values(
        [
            "split",
            "prompt_tokens",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


print()
print("=" * 100)
print("COMPLETELY TRUNCATED RECORDS")
print("=" * 100)

display(
    complete_df[
        [
            "split",
            "row_number",
            "ID",
            "General Category",
            "Category",
            "full_tokens",
            "prompt_tokens",
            "lcp_tokens",
            "prompt_is_exact_prefix",
            "assistant_region_tokens",
            "assistant_tokens_surviving_1024",
        ]
    ]
)


# --------------------------------------------------------------------------------------
# SPECIAL DIAGNOSTIC — RECORDS CLOSEST TO 1024 BOUNDARY
# --------------------------------------------------------------------------------------

borderline_df = (
    complete_df
    .assign(
        distance_from_1024=
        complete_df["prompt_tokens"]
        - MAX_SEQ_LENGTH
    )
    .sort_values(
        [
            "split",
            "distance_from_1024",
        ]
    )
)


print()
print("=" * 100)
print("BORDERLINE COMPLETELY-TRUNCATED RECORDS")
print("=" * 100)

display(
    borderline_df[
        [
            "split",
            "row_number",
            "ID",
            "full_tokens",
            "prompt_tokens",
            "distance_from_1024",
            "lcp_tokens",
            "prompt_is_exact_prefix",
        ]
    ].head(30)
)


# --------------------------------------------------------------------------------------
# SAVE
# --------------------------------------------------------------------------------------

audit_path = (
    REPORT_DIR
    / "cell22_exact_checkpoint_tokenizer_audit.csv"
)

summary_path = (
    REPORT_DIR
    / "cell22_exact_checkpoint_tokenizer_summary.json"
)

complete_path = (
    REPORT_DIR
    / "cell22_completely_truncated_records.csv"
)


audit_df.to_csv(
    audit_path,
    index=False,
)

complete_df.to_csv(
    complete_path,
    index=False,
)


result_summary = {
    "base_chat_template_sha256":
        base_template_hash,

    "checkpoint_chat_template_sha256":
        checkpoint_template_hash,

    "templates_identical":
        bool(
            base_tokenizer.chat_template
            == checkpoint_tokenizer.chat_template
        ),

    "max_seq_length":
        MAX_SEQ_LENGTH,

    "splits":
        summary,
}


with summary_path.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        result_summary,
        f,
        ensure_ascii=False,
        indent=2,
    )


print()
print("=" * 100)
print("CELL 22 COMPLETE")
print("=" * 100)

print("Full audit :", audit_path)
print("Complete   :", complete_path)
print("Summary    :", summary_path)

print()
print(
    "Do not start FT-10K yet. "
    "Review this summary first."
)

CELL 22 — EXACT FT-5K CHECKPOINT TOKENIZER / TEMPLATE AUDIT
TRAIN      : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_ft_005k_STRAT_sft_messages.jsonl
VALIDATION : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/imb_validation_2000_sft_messages.jsonl
CHECKPOINT : /home/eugrug/.unsloth/studio/outputs/unsloth_medgemma-1.5-4b-it-unsloth-bnb-4bit__project-emcrag_ft_imb_005k_v0701_1788215983/checkpoint-623

Checkpoint tokenizer class: GemmaTokenizerFast

CHAT TEMPLATE COMPARISON
Base template SHA256      : 7de1c58e208eda46e9c7f86397df37ec49883aeece39fb961e0a6b24088dd3c4
Checkpoint template SHA256: 42245274a9e05c6a52192104f7ef21575c5bf2f60e8506af3874cbed669dd150
Templates identical       : False

UNSLOTH ROW-BOUND METADATA
{
  "max_train_rows": null,
  "seed": 20260831
}

SOURCE DATASET COUNTS
TRAIN      : 5000
VALIDATION : 2000


EXACT CHECKPOINT TOKENIZER AUDIT SUMMARY

TRAIN_5K
-----------------------------------------------------------------------------

,split,row_number,ID,General Category,Category,full_tokens,prompt_tokens,lcp_tokens,prompt_is_exact_prefix,assistant_region_tokens,assistant_tokens_surviving_1024
0,TRAIN_5K,3347,668820,Salute Mentale,Psicologia,1284,1024,1024,True,260,0
1,TRAIN_5K,4016,168067,Medicina Generale e Chirurgia Generale,Malattie infettive,1279,1079,1079,True,200,0
2,TRAIN_5K,2681,153884,"Cardiologia, Sistema Circolatorio ed Ematologia",Ematologia,1194,1086,1086,True,108,0
3,TRAIN_5K,959,792621,Medicina Generale e Chirurgia Generale,Malattie infettive,1206,1089,1089,True,117,0
4,TRAIN_5K,2739,663627,Salute Mentale,Psicologia,1288,1118,1118,True,170,0
5,TRAIN_5K,1031,348051,"Oculistica, Otorinolaringoiatria, Odontoiatria...",Odontoiatria e odontostomatologia,1166,1137,1137,True,29,0
6,TRAIN_5K,4128,375340,Ortopedia e Sistema Muscoloscheletrico,Ortopedia,1486,1155,1155,True,331,0
7,TRAIN_5K,4761,110180,"Urologia, Andrologia e Salute Maschile",Nefrologia,1251,1176,1176,True,75,0
8,TRAIN_5K,3533,377064,Ortopedia e Sistema Muscoloscheletrico,Ortopedia,1316,1184,1184,True,132,0
9,TRAIN_5K,3929,524538,"Urologia, Andrologia e Salute Maschile",Urologia,2350,1206,1206,True,1144,0



BORDERLINE COMPLETELY-TRUNCATED RECORDS


,split,row_number,ID,full_tokens,prompt_tokens,distance_from_1024,lcp_tokens,prompt_is_exact_prefix
0,TRAIN_5K,3347,668820,1284,1024,0,1024,True
1,TRAIN_5K,4016,168067,1279,1079,55,1079,True
2,TRAIN_5K,2681,153884,1194,1086,62,1086,True
3,TRAIN_5K,959,792621,1206,1089,65,1089,True
4,TRAIN_5K,2739,663627,1288,1118,94,1118,True
5,TRAIN_5K,1031,348051,1166,1137,113,1137,True
6,TRAIN_5K,4128,375340,1486,1155,131,1155,True
7,TRAIN_5K,4761,110180,1251,1176,152,1176,True
8,TRAIN_5K,3533,377064,1316,1184,160,1184,True
9,TRAIN_5K,3929,524538,2350,1206,182,1206,True



CELL 22 COMPLETE
Full audit : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell22_exact_checkpoint_tokenizer_audit.csv
Complete   : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell22_completely_truncated_records.csv
Summary    : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell22_exact_checkpoint_tokenizer_summary.json

Do not start FT-10K yet. Review this summary first.


In [1]:
# ======================================================================================
# CELL 23 — STANDALONE DEV SANITY CHECK: BASE MEDGEMMA (M0) vs FT-5K (M1)
# ======================================================================================
#
# EMCRAG Fine-Tuning Experimental Protocol v07.01
#
# SCIENTIFIC PURPOSE
# ------------------
# This cell performs a SMALL, LOCAL, DEVELOPMENT-TIME sanity check comparing:
#
#   M0 = original/base MedGemma 1.5 4B-it
#   M1 = the SAME base model + FT-5K LoRA adapter (checkpoint-623)
#
# THIS IS NOT THE FINAL BENCHMARK.
# THIS IS NOT A RAGAS EVALUATION.
#
# Its purpose is to answer a deliberately limited question before spending
# many additional GPU-hours on FT-10K / FT-25K:
#
#   "Does FT-5K still behave normally, and has fine-tuning avoided obvious
#    qualitative degeneration?"
#
# The cell detects obvious development-time problems such as:
#
#   - empty answers;
#   - extremely short/non-responsive answers;
#   - pathological repetition;
#   - collapse toward a fixed response template;
#   - repeated saturation of max_new_tokens;
#   - gross qualitative degradation relative to the base model.
#
# WHY DEV AND NOT THE FROZEN TEST SET?
# ------------------------------------
# The fixed DEV-1000 split exists for development/model-selection diagnostics.
# The frozen 1,000-case benchmark remains untouched for the final scientific
# evaluation.
#
# DETERMINISTIC SAMPLE
# --------------------
# Two records are selected from each "General Category" in DEV, normally
# producing 20 cases. Selection uses fixed seed 20260831 and a stable SHA256
# ranking, so the SAME cases are selected every time this cell is rerun.
#
# FAIR M0 vs M1 COMPARISON
# ------------------------
# M0 and M1 receive:
#
#   - the same questions;
#   - the same system instruction;
#   - the same checkpoint tokenizer/chat template;
#   - the same deterministic greedy decoding;
#   - no RAG/retrieval;
#   - no CRAG routing;
#   - no web;
#   - no RAGAS;
#   - no OpenAI API.
#
# Cell 22 established that the checkpoint chat template differs from the
# original base-model template. The checkpoint template is therefore used
# for BOTH M0 and M1 so that prompt formatting is controlled and the main
# experimental difference is the LoRA adaptation.
#
# GPU SAFETY
# ----------
# M0 and M1 are NEVER resident in GPU memory simultaneously.
#
# Phase 1:
#     load M0 -> generate all DEV answers -> unload M0 -> clear CUDA cache
#
# Phase 2:
#     reload same base -> attach FT-5K adapter -> generate M1 answers
#     -> unload -> clear CUDA cache
#
# This is important on the local RTX 5070 Laptop with ~8 GiB VRAM.
#
# IMPORTANT — COMPLETELY STANDALONE / AUTONOMOUS
# -----------------------------------------------
# CELLS 1–22 DO NOT NEED TO BE EXECUTED.
#
# This cell can be launched after:
#
#   - restarting the Jupyter kernel;
#   - reopening VS Code/notebook;
#   - restarting WSL2;
#   - restarting Windows.
#
# It reads only persistent files already stored on disk:
#
# DEV:
#   /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/exports/
#   imb_dev_1000_sft_messages.jsonl
#
# FT-5K:
#   /home/eugrug/.unsloth/studio/outputs/
#   unsloth_medgemma-1.5-4b-it-unsloth-bnb-4bit__
#   project-emcrag_ft_imb_005k_v0701_1788215983/checkpoint-623
#
# It does NOT modify DEV, TRAIN, VALIDATION, frozen benchmark, model weights,
# adapter or checkpoint.
#
# It can therefore be safely rerun later.
#
# LOCAL TOKEN-F1
# --------------
# A simple lexical token-overlap F1 against the IMB reference is calculated
# only as a descriptive development heuristic.
#
# IT IS NOT:
#
#   - a medical correctness score;
#   - a semantic metric;
#   - evidence of statistical superiority;
#   - a replacement for the final benchmark.
#
# Manual inspection of M0/M1 answers remains essential.
#
# GO / NO-GO INTERPRETATION
# -------------------------
# PASS means only that no obvious systematic degeneration is found and that
# it is reasonable to proceed with FT-10K.
#
# PASS DOES NOT mean that FT-5K has already been proven superior to M0.
#
# ======================================================================================

from pathlib import Path
import json
import hashlib
import re
import time
import gc
from collections import Counter

import pandas as pd
import torch
from transformers import AutoTokenizer

try:
    from transformers import AutoModelForImageTextToText as AutoMedModel
except ImportError:
    try:
        from transformers import AutoModelForMultimodalLM as AutoMedModel
    except ImportError as exc:
        raise ImportError(
            "Transformers does not expose a compatible MedGemma multimodal "
            "model loader in this environment."
        ) from exc

try:
    from peft import PeftModel
except ImportError as exc:
    raise ImportError(
        "PEFT is required to load the FT-5K LoRA adapter."
    ) from exc


# --------------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------------

BASE_DIR = Path(
    "/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01"
)

DEV_JSONL = (
    BASE_DIR
    / "exports"
    / "imb_dev_1000_sft_messages.jsonl"
)

RUN_DIR = Path(
    "/home/eugrug/.unsloth/studio/outputs/"
    "unsloth_medgemma-1.5-4b-it-unsloth-bnb-4bit"
    "__project-emcrag_ft_imb_005k_v0701_1788215983"
)

CHECKPOINT_DIR = RUN_DIR / "checkpoint-623"
ADAPTER_CONFIG = CHECKPOINT_DIR / "adapter_config.json"

REPORT_DIR = BASE_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260831

# 2 examples x 10 General Categories = normally 20 DEV cases.
N_PER_GENERAL_CATEGORY = 2

# Deliberately limited because this is a sanity check, not final generation.
MAX_NEW_TOKENS = 256

# Deterministic greedy decoding.
DO_SAMPLE = False

FALLBACK_BASE_MODEL_ID = (
    "unsloth/medgemma-1.5-4b-it-bnb-4bit"
)


# --------------------------------------------------------------------------------------
# PRE-FLIGHT
# --------------------------------------------------------------------------------------

print("=" * 110)
print("CELL 23 — DEV SANITY CHECK: M0 BASE vs M1 FT-5K")
print("=" * 110)

for p in [DEV_JSONL, CHECKPOINT_DIR, ADAPTER_CONFIG]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required artifact: {p}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available in this notebook kernel."
    )

print("GPU            :", torch.cuda.get_device_name(0))
print(
    "VRAM           :",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB",
)
print("BF16 supported :", torch.cuda.is_bf16_supported())


# --------------------------------------------------------------------------------------
# RECOVER THE EXACT BASE MODEL FROM THE ADAPTER CONFIG
# --------------------------------------------------------------------------------------

with ADAPTER_CONFIG.open("r", encoding="utf-8") as f:
    adapter_cfg = json.load(f)

BASE_MODEL_ID = adapter_cfg.get("base_model_name_or_path")

if not BASE_MODEL_ID:
    BASE_MODEL_ID = FALLBACK_BASE_MODEL_ID
    print(
        "WARNING: base_model_name_or_path missing; fallback:",
        BASE_MODEL_ID,
    )

print("Base model     :", BASE_MODEL_ID)
print("FT checkpoint  :", CHECKPOINT_DIR)


# --------------------------------------------------------------------------------------
# USE EXACT CHECKPOINT TOKENIZER/TEMPLATE FOR BOTH M0 AND M1
# --------------------------------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    str(CHECKPOINT_DIR),
    use_fast=True,
)

if tokenizer.chat_template is None:
    raise RuntimeError(
        "Checkpoint tokenizer has no chat_template."
    )

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

template_hash = hashlib.sha256(
    tokenizer.chat_template.encode("utf-8")
).hexdigest()

print("Tokenizer      :", tokenizer.__class__.__name__)
print("Template SHA256:", template_hash)


# --------------------------------------------------------------------------------------
# LOAD DEV
# --------------------------------------------------------------------------------------

def load_jsonl(path):
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(
                    f"Invalid JSON at line {line_no}: {path}"
                ) from exc

    return rows


dev_records = load_jsonl(DEV_JSONL)

print("DEV records    :", len(dev_records))

if len(dev_records) != 1000:
    print(
        f"WARNING: protocol expects DEV=1000; found {len(dev_records)}."
    )


# --------------------------------------------------------------------------------------
# NORMALIZE DEV
# --------------------------------------------------------------------------------------

def extract_row(record, source_row):
    messages = record.get("messages", [])
    metadata = record.get("metadata", {}) or {}

    if (
        len(messages) != 3
        or messages[0].get("role") != "system"
        or messages[1].get("role") != "user"
        or messages[2].get("role") != "assistant"
    ):
        raise ValueError(
            f"Unexpected DEV structure at row {source_row}"
        )

    return {
        "source_row": source_row,
        "ID": metadata.get("ID"),
        "General Category": metadata.get("General Category"),
        "Category": metadata.get("Category"),
        "system": str(
            messages[0].get("content", "") or ""
        ).strip(),
        "question": str(
            messages[1].get("content", "") or ""
        ).strip(),
        "reference_answer": str(
            messages[2].get("content", "") or ""
        ).strip(),
    }


dev_df = pd.DataFrame(
    [
        extract_row(record, i)
        for i, record in enumerate(dev_records)
    ]
)

if dev_df["question"].eq("").any():
    raise RuntimeError("Empty DEV question detected.")

if dev_df["reference_answer"].eq("").any():
    raise RuntimeError("Empty DEV reference answer detected.")


# --------------------------------------------------------------------------------------
# DETERMINISTIC STRATIFIED DEV SELECTION
# --------------------------------------------------------------------------------------

def stable_rank(row):
    key = (
        f"{SEED}|{row['ID']}|{row['source_row']}|"
        f"{row['General Category']}"
    )

    return hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()


dev_df["_rank"] = dev_df.apply(
    stable_rank,
    axis=1,
)

selected = []

for general_category, group in dev_df.groupby(
    "General Category",
    dropna=False,
    sort=True,
):
    g = group.sort_values(
        ["_rank", "source_row"]
    )

    selected.append(
        g.head(N_PER_GENERAL_CATEGORY)
    )

selected_df = (
    pd.concat(selected, ignore_index=True)
    .sort_values(["General Category", "_rank"])
    .reset_index(drop=True)
)

selected_df.insert(
    0,
    "sanity_case",
    range(1, len(selected_df) + 1),
)

print()
print("=" * 110)
print("DETERMINISTIC DEV SELECTION")
print("=" * 110)
print("Selected cases:", len(selected_df))
print()
print(
    selected_df["General Category"]
    .value_counts(sort=False)
    .sort_index()
)


# Persist selected IDs independently of generated answers.

SELECTION_PATH = (
    REPORT_DIR
    / "cell23_dev_sanity_selection.json"
)

selection_payload = {
    "protocol_version": "v07.01",
    "seed": SEED,
    "n_per_general_category": N_PER_GENERAL_CATEGORY,
    "cases": [
        {
            "sanity_case": int(row["sanity_case"]),
            "source_row": int(row["source_row"]),
            "ID": None if pd.isna(row["ID"]) else row["ID"],
            "General Category": row["General Category"],
            "Category": row["Category"],
        }
        for _, row in selected_df.iterrows()
    ],
}

with SELECTION_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        selection_payload,
        f,
        ensure_ascii=False,
        indent=2,
    )


# --------------------------------------------------------------------------------------
# MODEL / GENERATION HELPERS
# --------------------------------------------------------------------------------------

def load_base_model():

    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )

    model = AutoMedModel.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )

    model.eval()
    return model


def get_input_device(model):

    try:
        if str(model.device) != "meta":
            return model.device
    except Exception:
        pass

    for parameter in model.parameters():
        if parameter.device.type == "cuda":
            return parameter.device

    return torch.device("cuda:0")


def make_prompt(system_text, question):

    messages = [
        {
            "role": "system",
            "content": system_text,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


@torch.inference_mode()
def generate_one(model, system_text, question):

    prompt = make_prompt(
        system_text,
        question,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    )

    device = get_input_device(model)

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    input_tokens = inputs["input_ids"].shape[-1]

    torch.cuda.synchronize()
    start = time.perf_counter()

    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=DO_SAMPLE,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    generated_ids = output[0, input_tokens:]

    generated_tokens = int(
        generated_ids.shape[-1]
    )

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "answer": answer,
        "input_tokens": int(input_tokens),
        "generated_tokens": generated_tokens,
        "hit_max_new_tokens": (
            generated_tokens >= MAX_NEW_TOKENS
        ),
        "seconds": elapsed,
        "tokens_per_second": (
            generated_tokens / elapsed
            if elapsed > 0 else None
        ),
    }


def clear_gpu():

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


# --------------------------------------------------------------------------------------
# PHASE 1 — M0 BASE
# --------------------------------------------------------------------------------------

print()
print("=" * 110)
print("PHASE 1/2 — M0 BASE MEDGEMMA")
print("=" * 110)

m0_model = load_base_model()

print(
    "CUDA allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB",
)

m0_results = []

for i, row in selected_df.iterrows():

    result = generate_one(
        m0_model,
        row["system"],
        row["question"],
    )

    m0_results.append(result)

    print(
        f"M0 {i+1:02d}/{len(selected_df):02d} | "
        f"ID={row['ID']} | "
        f"{result['generated_tokens']} tok | "
        f"{result['seconds']:.1f}s"
    )


del m0_model
clear_gpu()

print(
    "CUDA allocated after M0 unload:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB",
)


# --------------------------------------------------------------------------------------
# PHASE 2 — M1 = BASE + FT-5K ADAPTER
# --------------------------------------------------------------------------------------

print()
print("=" * 110)
print("PHASE 2/2 — M1 FT-5K")
print("=" * 110)

m1_base = load_base_model()

m1_model = PeftModel.from_pretrained(
    m1_base,
    str(CHECKPOINT_DIR),
    is_trainable=False,
)

m1_model.eval()

print(
    "CUDA allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB",
)

m1_results = []

for i, row in selected_df.iterrows():

    result = generate_one(
        m1_model,
        row["system"],
        row["question"],
    )

    m1_results.append(result)

    print(
        f"M1 {i+1:02d}/{len(selected_df):02d} | "
        f"ID={row['ID']} | "
        f"{result['generated_tokens']} tok | "
        f"{result['seconds']:.1f}s"
    )


del m1_model
del m1_base
clear_gpu()

print(
    "CUDA allocated after M1 unload:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GiB",
)


# --------------------------------------------------------------------------------------
# DESCRIPTIVE LOCAL HEURISTICS
# --------------------------------------------------------------------------------------

word_re = re.compile(
    r"\b[\wÀ-ÿ]+\b",
    flags=re.UNICODE,
)


def normalize_tokens(text):
    return [
        token.lower()
        for token in word_re.findall(text or "")
    ]


def token_f1(candidate, reference):
    """
    Lexical overlap ONLY.
    This is NOT a medical correctness metric.
    """

    cand = normalize_tokens(candidate)
    ref = normalize_tokens(reference)

    if not cand and not ref:
        return 1.0

    if not cand or not ref:
        return 0.0

    cand_counts = Counter(cand)
    ref_counts = Counter(ref)

    overlap = sum(
        (cand_counts & ref_counts).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(cand)
    recall = overlap / len(ref)

    return (
        2 * precision * recall
        / (precision + recall)
    )


def repeated_4gram_ratio(text):

    tokens = normalize_tokens(text)

    if len(tokens) < 8:
        return 0.0

    grams = [
        tuple(tokens[i:i + 4])
        for i in range(len(tokens) - 3)
    ]

    return 1.0 - (
        len(set(grams))
        / len(grams)
    )


# --------------------------------------------------------------------------------------
# RESULT TABLE
# --------------------------------------------------------------------------------------

result_df = selected_df[
    [
        "sanity_case",
        "source_row",
        "ID",
        "General Category",
        "Category",
        "question",
        "reference_answer",
    ]
].copy()

result_df["m0_answer"] = [
    x["answer"] for x in m0_results
]

result_df["m1_ft5k_answer"] = [
    x["answer"] for x in m1_results
]

for prefix, results in [
    ("m0", m0_results),
    ("m1", m1_results),
]:

    result_df[f"{prefix}_generated_tokens"] = [
        x["generated_tokens"] for x in results
    ]

    result_df[f"{prefix}_hit_max_new_tokens"] = [
        x["hit_max_new_tokens"] for x in results
    ]

    result_df[f"{prefix}_generation_seconds"] = [
        x["seconds"] for x in results
    ]

    result_df[f"{prefix}_tokens_per_second"] = [
        x["tokens_per_second"] for x in results
    ]


result_df["m0_reference_token_f1"] = [
    token_f1(candidate, reference)
    for candidate, reference in zip(
        result_df["m0_answer"],
        result_df["reference_answer"],
    )
]

result_df["m1_reference_token_f1"] = [
    token_f1(candidate, reference)
    for candidate, reference in zip(
        result_df["m1_ft5k_answer"],
        result_df["reference_answer"],
    )
]

result_df["delta_token_f1_m1_minus_m0"] = (
    result_df["m1_reference_token_f1"]
    - result_df["m0_reference_token_f1"]
)

result_df["m0_empty"] = (
    result_df["m0_answer"].str.strip().eq("")
)

result_df["m1_empty"] = (
    result_df["m1_ft5k_answer"].str.strip().eq("")
)

result_df["m0_very_short"] = (
    result_df["m0_answer"].str.len() < 20
)

result_df["m1_very_short"] = (
    result_df["m1_ft5k_answer"].str.len() < 20
)

result_df["m0_repeated_4gram_ratio"] = [
    repeated_4gram_ratio(x)
    for x in result_df["m0_answer"]
]

result_df["m1_repeated_4gram_ratio"] = [
    repeated_4gram_ratio(x)
    for x in result_df["m1_ft5k_answer"]
]


# --------------------------------------------------------------------------------------
# SUMMARY
# --------------------------------------------------------------------------------------

summary = {
    "protocol_version": "v07.01",
    "seed": SEED,
    "selected_dev_cases": int(len(result_df)),
    "base_model_id": BASE_MODEL_ID,
    "ft_checkpoint": str(CHECKPOINT_DIR),
    "common_template_sha256": template_hash,
    "max_new_tokens": MAX_NEW_TOKENS,
    "do_sample": DO_SAMPLE,

    "M0": {
        "empty": int(result_df["m0_empty"].sum()),
        "very_short": int(
            result_df["m0_very_short"].sum()
        ),
        "hit_max_new_tokens": int(
            result_df["m0_hit_max_new_tokens"].sum()
        ),
        "mean_reference_token_f1": float(
            result_df["m0_reference_token_f1"].mean()
        ),
        "mean_generated_tokens": float(
            result_df["m0_generated_tokens"].mean()
        ),
    },

    "M1_FT5K": {
        "empty": int(result_df["m1_empty"].sum()),
        "very_short": int(
            result_df["m1_very_short"].sum()
        ),
        "hit_max_new_tokens": int(
            result_df["m1_hit_max_new_tokens"].sum()
        ),
        "mean_reference_token_f1": float(
            result_df["m1_reference_token_f1"].mean()
        ),
        "mean_generated_tokens": float(
            result_df["m1_generated_tokens"].mean()
        ),
    },

    "mean_token_f1_delta_M1_minus_M0": float(
        result_df[
            "delta_token_f1_m1_minus_m0"
        ].mean()
    ),

    "warning": (
        "Token-F1 is descriptive lexical overlap only; "
        "it is not a medical correctness metric."
    ),
}


print()
print("=" * 110)
print("CELL 23 — SANITY CHECK SUMMARY")
print("=" * 110)

print("Selected DEV cases:", len(result_df))

print("\nM0 BASE")
print("  Empty                :", summary["M0"]["empty"])
print("  Very short           :", summary["M0"]["very_short"])
print(
    "  Hit max_new_tokens   :",
    summary["M0"]["hit_max_new_tokens"],
)
print(
    "  Mean lexical token-F1:",
    f"{summary['M0']['mean_reference_token_f1']:.4f}",
)

print("\nM1 FT-5K")
print("  Empty                :", summary["M1_FT5K"]["empty"])
print("  Very short           :", summary["M1_FT5K"]["very_short"])
print(
    "  Hit max_new_tokens   :",
    summary["M1_FT5K"]["hit_max_new_tokens"],
)
print(
    "  Mean lexical token-F1:",
    f"{summary['M1_FT5K']['mean_reference_token_f1']:.4f}",
)

print(
    "\nMean token-F1 delta M1-M0:",
    f"{summary['mean_token_f1_delta_M1_minus_M0']:+.4f}",
)

print(
    "\nIMPORTANT: manual M0/M1 answer review remains mandatory."
)


# --------------------------------------------------------------------------------------
# SAVE
# --------------------------------------------------------------------------------------

CSV_PATH = (
    REPORT_DIR
    / "cell23_dev_sanity_m0_vs_ft5k.csv"
)

JSON_PATH = (
    REPORT_DIR
    / "cell23_dev_sanity_summary.json"
)

result_df.to_csv(
    CSV_PATH,
    index=False,
)

with JSON_PATH.open("w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2,
    )


# --------------------------------------------------------------------------------------
# MANUAL REVIEW TABLE
# --------------------------------------------------------------------------------------

pd.set_option(
    "display.max_colwidth",
    180,
)

display(
    result_df[
        [
            "sanity_case",
            "ID",
            "General Category",
            "Category",
            "question",
            "reference_answer",
            "m0_answer",
            "m1_ft5k_answer",
            "m0_reference_token_f1",
            "m1_reference_token_f1",
            "delta_token_f1_m1_minus_m0",
        ]
    ]
)

print()
print("=" * 110)
print("CELL 23 COMPLETE")
print("=" * 110)

print("Selection :", SELECTION_PATH)
print("Comparison:", CSV_PATH)
print("Summary   :", JSON_PATH)

print()
print(
    "NEXT: review these results before authorizing FT-10K."
)

In [2]:
# ======================================================================================
# CELL 24 v2 — STANDALONE IMB REFERENCE QUALITY AUDIT SAMPLE
#              DETERMINISTIC PROPORTIONAL STRATIFIED SELECTION — N = 500
# ======================================================================================
#
# EMCRAG Fine-Tuning Experimental Protocol v07.01
#
# WHY THIS CELL EXISTS
# --------------------
# Cell 23 revealed a potentially important methodological limitation:
#
#     IMB "Answer" cannot automatically be assumed to be a uniformly
#     reliable clinical gold-standard answer.
#
# Some DEV examples were incomplete, conversationally incidental,
# weakly informative or unsuitable as authoritative clinical references.
#
# The 20 DEV cases are sufficient to DISCOVER this problem, but not to
# estimate its prevalence in IMB.
#
# Cell 24 therefore freezes an independent deterministic sample of 500
# records from the TRAIN POOL for a dedicated reference-quality audit.
#
#
# IMPORTANT — CELL 24 IS STANDALONE / AUTONOMOUS
# -----------------------------------------------
#
# CELLS 1–23 DO NOT NEED TO BE EXECUTED.
#
# It can be run:
#
#   - after a fresh Jupyter kernel;
#   - after reopening VS Code;
#   - after restarting WSL2;
#   - after restarting Windows.
#
# It reads only persistent artifacts already saved on disk.
#
# It does NOT:
#
#   - train MedGemma;
#   - use the GPU;
#   - modify model weights;
#   - modify TRAIN/DEV/VALIDATION/TEST;
#   - call Unsloth Studio;
#   - call OpenAI;
#   - call RAGAS;
#   - inspect the frozen benchmark.
#
# It can safely be rerun.
#
#
# IMPORTANT CORRECTION FROM CELL 24 v1
# -------------------------------------
#
# The full IMB source directory:
#
#   /mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe
#
# contains Arrow data, but it is NOT necessarily a Hugging Face Dataset
# directory created with Dataset.save_to_disk().
#
# Therefore:
#
#       load_from_disk(FULL_IMB_PATH)
#
# is NOT assumed to work.
#
# Cell 24 v2 first uses the persistent TRAIN manifest/order created during
# the v07.01 data-preparation phase:
#
#   imb_train_nested_order.parquet
#
# This artifact already represents the frozen TRAIN pool.
#
# If Question and Answer are already present there, no access to the large
# source dataset is required.
#
# If only identities/categories are present, the cell:
#
#   1. selects the 500 audit IDs FIRST;
#   2. discovers .arrow/.parquet source shards;
#   3. reads only enough source data to resolve those 500 IDs.
#
# It never needs to convert the entire ~8 GB IMB corpus to Pandas.
#
#
# SAMPLING DESIGN
# ---------------
#
# Sampling unit:
#
#     one IMB Question/Answer record
#
# Population:
#
#     frozen TRAIN POOL only
#
# Excluded:
#
#     DEV
#     VALIDATION
#     frozen TEST
#
# Stratification:
#
#     General Category
#
# Allocation:
#
#     proportional allocation using largest remainder
#
# Sample size:
#
#     N = 500
#
# Fixed seed:
#
#     20260831
#
# Within each stratum, records are ranked deterministically using:
#
#     SHA256(seed | ID | General Category)
#
# Therefore the same source/train split always produces the same 500 IDs.
#
#
# THIS CELL DOES NOT ASSIGN QUALITY CLASSES
# -----------------------------------------
#
# The following review columns are intentionally left empty:
#
#     Quality_Class
#     Clinical_Relevance
#     Completeness
#     Potential_Error
#     Notes
#
# Planned taxonomy for the NEXT stage:
#
#     A — clinically substantive
#     B — partially useful
#     C — weak reference
#     D — unsuitable as clinical ground truth
#
#
# OUTPUTS
# -------
#
# reports/imb_reference_quality_audit_500.csv
# reports/imb_reference_quality_audit_500.parquet
# reports/imb_reference_quality_audit_500_metadata.json
#
# The metadata JSON contains a SHA256 fingerprint of the selected IDs.
#
# That fingerprint freezes the exact audit sample for reproducibility and
# possible reporting in the final paper.
#
# ======================================================================================


from pathlib import Path
import hashlib
import json
import math
import gc

import pandas as pd


# ======================================================================================
# CONFIG
# ======================================================================================

SEED = 20260831
AUDIT_N = 500

BASE_DIR = Path(
    "/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01"
)

# Primary frozen TRAIN artifact.
TRAIN_ORDER_PATH = (
    BASE_DIR
    / "manifests"
    / "imb_train_nested_order.parquet"
)

# Secondary split manifest, useful for diagnostics/fallback.
SPLIT_MANIFEST_PATH = (
    BASE_DIR
    / "manifests"
    / "imb_split_manifest.parquet"
)

# Original large IMB source.
FULL_IMB_PATH = Path(
    "/mnt/f/processed_datasets/IMB_Dataset_Arrow/IMB_Dataset_Safe"
)

REPORT_DIR = BASE_DIR / "reports"
REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUTPUT_CSV = (
    REPORT_DIR
    / "imb_reference_quality_audit_500.csv"
)

OUTPUT_PARQUET = (
    REPORT_DIR
    / "imb_reference_quality_audit_500.parquet"
)

OUTPUT_META = (
    REPORT_DIR
    / "imb_reference_quality_audit_500_metadata.json"
)


# ======================================================================================
# START
# ======================================================================================

print("=" * 110)
print("CELL 24 v2 — IMB REFERENCE QUALITY AUDIT SAMPLE — N=500")
print("=" * 110)

print("Seed                :", SEED)
print("Requested audit N   :", AUDIT_N)
print("TRAIN order manifest:", TRAIN_ORDER_PATH)
print("Split manifest      :", SPLIT_MANIFEST_PATH)
print("Full IMB source     :", FULL_IMB_PATH)
print()


# ======================================================================================
# BASIC CHECKS
# ======================================================================================

if not TRAIN_ORDER_PATH.exists():
    raise FileNotFoundError(
        "Frozen TRAIN order manifest not found:\n"
        f"{TRAIN_ORDER_PATH}"
    )

if not FULL_IMB_PATH.exists():
    raise FileNotFoundError(
        "Full IMB source path not found:\n"
        f"{FULL_IMB_PATH}"
    )


# ======================================================================================
# HELPER — COLUMN RESOLUTION
# ======================================================================================

def find_column(
    df,
    candidates,
    required=True,
):
    """
    Resolve column names case-insensitively using a controlled alias list.
    """

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.strip().lower()

        if key in lookup:
            return lookup[key]

    if required:
        raise KeyError(
            "Required column not found.\n"
            f"Tried: {candidates}\n"
            f"Available columns: {list(df.columns)}"
        )

    return None


# ======================================================================================
# LOAD FROZEN TRAIN ORDER
# ======================================================================================

print("=" * 110)
print("LOAD FROZEN TRAIN ORDER")
print("=" * 110)

train_source = pd.read_parquet(
    TRAIN_ORDER_PATH
)

print(
    "Rows   :",
    f"{len(train_source):,}",
)

print(
    "Columns:",
    list(train_source.columns),
)

print()


# ======================================================================================
# IDENTIFY AVAILABLE FIELDS
# ======================================================================================

id_col = find_column(
    train_source,
    [
        "ID",
        "id",
        "source_id",
        "record_id",
        "imb_id",
    ],
)

general_col = find_column(
    train_source,
    [
        "General Category",
        "general_category",
        "general category",
    ],
    required=False,
)

category_col = find_column(
    train_source,
    [
        "Category",
        "category",
    ],
    required=False,
)

question_col = find_column(
    train_source,
    [
        "Question",
        "question",
    ],
    required=False,
)

answer_col = find_column(
    train_source,
    [
        "Answer",
        "answer",
    ],
    required=False,
)


# ======================================================================================
# IF GENERAL CATEGORY IS MISSING, TRY SPLIT MANIFEST
# ======================================================================================

if general_col is None:

    print(
        "General Category not present in TRAIN order manifest."
    )

    print(
        "Attempting recovery from split manifest..."
    )

    if not SPLIT_MANIFEST_PATH.exists():
        raise FileNotFoundError(
            "General Category missing and split manifest "
            "is unavailable."
        )

    split_df = pd.read_parquet(
        SPLIT_MANIFEST_PATH
    )

    print(
        "Split manifest columns:",
        list(split_df.columns),
    )

    split_id_col = find_column(
        split_df,
        [
            "ID",
            "id",
            "source_id",
            "record_id",
            "imb_id",
        ],
    )

    split_general_col = find_column(
        split_df,
        [
            "General Category",
            "general_category",
            "general category",
        ],
    )

    # Normalize join key.
    train_source["_join_id"] = (
        train_source[id_col]
        .astype(str)
        .str.strip()
    )

    split_df["_join_id"] = (
        split_df[split_id_col]
        .astype(str)
        .str.strip()
    )

    general_map = (
        split_df[
            [
                "_join_id",
                split_general_col,
            ]
        ]
        .drop_duplicates("_join_id")
    )

    train_source = train_source.merge(
        general_map,
        on="_join_id",
        how="left",
        validate="one_to_one",
    )

    general_col = split_general_col

    if train_source[
        general_col
    ].isna().any():

        raise RuntimeError(
            "Could not recover General Category "
            "for every TRAIN record."
        )

    del split_df
    del general_map
    gc.collect()


# ======================================================================================
# NORMALIZE SAMPLING FRAME
# ======================================================================================

sampling_df = pd.DataFrame(
    {
        "ID":
            train_source[id_col],

        "General Category":
            train_source[general_col],
    }
)


if category_col is not None:
    sampling_df["Category"] = (
        train_source[category_col]
    )

if question_col is not None:
    sampling_df["Question"] = (
        train_source[question_col]
    )

if answer_col is not None:
    sampling_df["Answer"] = (
        train_source[answer_col]
    )


sampling_df["ID_key"] = (
    sampling_df["ID"]
    .astype(str)
    .str.strip()
)


# ======================================================================================
# TRAIN POOL VALIDATION
# ======================================================================================

print("=" * 110)
print("TRAIN POOL VALIDATION")
print("=" * 110)

print(
    "TRAIN records:",
    f"{len(sampling_df):,}",
)

print(
    "Unique IDs   :",
    f"{sampling_df['ID_key'].nunique():,}",
)

EXPECTED_TRAIN_POOL = 779_192

if len(sampling_df) != EXPECTED_TRAIN_POOL:

    print(
        "WARNING: protocol expectation is "
        f"{EXPECTED_TRAIN_POOL:,}, but current TRAIN artifact "
        f"contains {len(sampling_df):,} records."
    )


if sampling_df["ID_key"].duplicated().any():

    dup_n = int(
        sampling_df["ID_key"]
        .duplicated()
        .sum()
    )

    raise RuntimeError(
        f"Duplicated TRAIN identities detected: {dup_n}"
    )


if sampling_df[
    "General Category"
].isna().any():

    raise RuntimeError(
        "Missing General Category values detected."
    )


print(
    "General Categories:",
    sampling_df[
        "General Category"
    ].nunique(),
)

print()


# ======================================================================================
# CATEGORY DISTRIBUTION
# ======================================================================================

counts = (
    sampling_df[
        "General Category"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

category_table = pd.DataFrame(
    {
        "General Category":
            counts.index,

        "Train_Count":
            counts.values,
    }
)

category_table[
    "Train_Proportion"
] = (
    category_table[
        "Train_Count"
    ]
    / len(sampling_df)
)


# ======================================================================================
# LARGEST-REMAINDER PROPORTIONAL ALLOCATION
# ======================================================================================

category_table[
    "Exact_Quota"
] = (
    category_table[
        "Train_Proportion"
    ]
    * AUDIT_N
)

category_table[
    "Quota"
] = (
    category_table[
        "Exact_Quota"
    ]
    .apply(math.floor)
    .astype(int)
)

category_table[
    "Remainder"
] = (
    category_table[
        "Exact_Quota"
    ]
    - category_table[
        "Quota"
    ]
)

remaining = (
    AUDIT_N
    - int(
        category_table[
            "Quota"
        ].sum()
    )
)


if remaining > 0:

    winners = (
        category_table
        .sort_values(
            [
                "Remainder",
                "General Category",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .index[:remaining]
    )

    category_table.loc[
        winners,
        "Quota",
    ] += 1


if int(
    category_table[
        "Quota"
    ].sum()
) != AUDIT_N:

    raise RuntimeError(
        "Quota allocation did not produce "
        f"exactly {AUDIT_N} records."
    )


print("=" * 110)
print("PROPORTIONAL STRATIFICATION QUOTAS")
print("=" * 110)

display(
    category_table[
        [
            "General Category",
            "Train_Count",
            "Train_Proportion",
            "Quota",
        ]
    ]
)


# ======================================================================================
# DETERMINISTIC SHA256 RANK
# ======================================================================================

def stable_rank(row):

    key = (
        f"{SEED}|"
        f"{row['ID_key']}|"
        f"{row['General Category']}"
    )

    return hashlib.sha256(
        key.encode("utf-8")
    ).hexdigest()


sampling_df[
    "_audit_rank"
] = sampling_df.apply(
    stable_rank,
    axis=1,
)


# ======================================================================================
# SELECT EXACTLY 500 RECORDS
# ======================================================================================

quota_lookup = dict(
    zip(
        category_table[
            "General Category"
        ],
        category_table[
            "Quota"
        ],
    )
)

parts = []


for general_category, group in sampling_df.groupby(
    "General Category",
    sort=True,
):

    quota = int(
        quota_lookup[
            general_category
        ]
    )

    ranked = group.sort_values(
        [
            "_audit_rank",
            "ID_key",
        ]
    )

    if len(ranked) < quota:

        raise RuntimeError(
            f"Category '{general_category}' "
            f"contains {len(ranked)} records "
            f"but quota={quota}."
        )

    parts.append(
        ranked.head(quota)
    )


audit_df = pd.concat(
    parts,
    ignore_index=True,
)

audit_df = (
    audit_df
    .sort_values(
        [
            "General Category",
            "_audit_rank",
        ]
    )
    .reset_index(drop=True)
)


if len(audit_df) != AUDIT_N:

    raise RuntimeError(
        f"Expected {AUDIT_N} selected records; "
        f"got {len(audit_df)}."
    )


if audit_df[
    "ID_key"
].nunique() != AUDIT_N:

    raise RuntimeError(
        "Selected audit sample contains duplicate IDs."
    )


# ======================================================================================
# CHECK WHETHER QUESTION / ANSWER ARE ALREADY AVAILABLE
# ======================================================================================

has_question = (
    "Question"
    in audit_df.columns
)

has_answer = (
    "Answer"
    in audit_df.columns
)

has_category = (
    "Category"
    in audit_df.columns
)


print()
print("=" * 110)
print("QA FIELD AVAILABILITY")
print("=" * 110)

print(
    "Question already in TRAIN artifact:",
    has_question,
)

print(
    "Answer already in TRAIN artifact  :",
    has_answer,
)

print(
    "Category already in TRAIN artifact:",
    has_category,
)

print()


# ======================================================================================
# SOURCE RESOLUTION HELPER
# ======================================================================================

def discover_source_files(root):

    arrow_files = sorted(
        root.rglob("*.arrow")
    )

    parquet_files = sorted(
        root.rglob("*.parquet")
    )

    feather_files = sorted(
        root.rglob("*.feather")
    )

    return (
        arrow_files,
        parquet_files,
        feather_files,
    )


# ======================================================================================
# RESOLVE QA FROM ORIGINAL DATA ONLY IF REQUIRED
# ======================================================================================

need_source_lookup = (
    not has_question
    or not has_answer
    or not has_category
)


if need_source_lookup:

    print("=" * 110)
    print("RESOLVING 500 SELECTED IDs FROM ORIGINAL IMB SOURCE")
    print("=" * 110)

    selected_ids = set(
        audit_df[
            "ID_key"
        ].tolist()
    )

    (
        arrow_files,
        parquet_files,
        feather_files,
    ) = discover_source_files(
        FULL_IMB_PATH
    )


    print(
        "Arrow files  :",
        len(arrow_files),
    )

    print(
        "Parquet files:",
        len(parquet_files),
    )

    print(
        "Feather files:",
        len(feather_files),
    )


    if (
        not arrow_files
        and not parquet_files
        and not feather_files
    ):

        print()
        print(
            "Directory contents (first 100 paths):"
        )

        for p in list(
            FULL_IMB_PATH.rglob("*")
        )[:100]:

            print(" ", p)

        raise RuntimeError(
            "No .arrow, .parquet or .feather "
            "source files were found."
        )


    resolved_parts = []


    # ------------------------------------------------------------------
    # A) HUGGING FACE / APACHE ARROW FILES
    # ------------------------------------------------------------------

    if arrow_files:

        from datasets import Dataset


        for file_no, arrow_path in enumerate(
            arrow_files,
            start=1,
        ):

            print(
                f"Scanning Arrow "
                f"{file_no}/{len(arrow_files)}: "
                f"{arrow_path.name}"
            )

            try:

                ds = Dataset.from_file(
                    str(arrow_path)
                )

            except Exception as exc:

                print(
                    "  WARNING: Dataset.from_file failed:",
                    repr(exc),
                )

                continue


            required = {
                "ID",
                "General Category",
                "Category",
                "Question",
                "Answer",
            }


            if not required.issubset(
                set(ds.column_names)
            ):

                print(
                    "  Skipped: required columns missing."
                )

                print(
                    "  Columns:",
                    ds.column_names,
                )

                del ds
                continue


            # Filter only the 500 identities.
            subset = ds.filter(
                lambda batch: [
                    str(x).strip()
                    in selected_ids
                    for x in batch["ID"]
                ],
                batched=True,
                batch_size=10_000,
            )


            if len(subset):

                part = (
                    subset
                    .select_columns(
                        [
                            "ID",
                            "General Category",
                            "Category",
                            "Question",
                            "Answer",
                        ]
                    )
                    .to_pandas()
                )

                resolved_parts.append(
                    part
                )

                print(
                    "  matched:",
                    len(part),
                )


            del subset
            del ds
            gc.collect()


    # ------------------------------------------------------------------
    # B) PARQUET FALLBACK
    # ------------------------------------------------------------------

    for file_no, pq_path in enumerate(
        parquet_files,
        start=1,
    ):

        already_found = set()

        if resolved_parts:

            already_found = set(
                pd.concat(
                    resolved_parts,
                    ignore_index=True,
                )["ID"]
                .astype(str)
                .str.strip()
            )


        remaining_ids = (
            selected_ids
            - already_found
        )


        if not remaining_ids:
            break


        print(
            f"Scanning Parquet "
            f"{file_no}/{len(parquet_files)}: "
            f"{pq_path.name}"
        )


        try:

            pq = pd.read_parquet(
                pq_path,
                columns=[
                    "ID",
                    "General Category",
                    "Category",
                    "Question",
                    "Answer",
                ],
            )

        except Exception as exc:

            print(
                "  WARNING:",
                repr(exc),
            )

            continue


        mask = (
            pq["ID"]
            .astype(str)
            .str.strip()
            .isin(remaining_ids)
        )


        part = pq.loc[
            mask,
            [
                "ID",
                "General Category",
                "Category",
                "Question",
                "Answer",
            ],
        ]


        if len(part):

            resolved_parts.append(
                part.copy()
            )

            print(
                "  matched:",
                len(part),
            )


        del pq
        gc.collect()


    # ------------------------------------------------------------------
    # C) FEATHER FALLBACK
    # ------------------------------------------------------------------

    for file_no, feather_path in enumerate(
        feather_files,
        start=1,
    ):

        already_found = set()

        if resolved_parts:

            already_found = set(
                pd.concat(
                    resolved_parts,
                    ignore_index=True,
                )["ID"]
                .astype(str)
                .str.strip()
            )


        remaining_ids = (
            selected_ids
            - already_found
        )


        if not remaining_ids:
            break


        print(
            f"Scanning Feather "
            f"{file_no}/{len(feather_files)}: "
            f"{feather_path.name}"
        )


        try:

            ft = pd.read_feather(
                feather_path,
                columns=[
                    "ID",
                    "General Category",
                    "Category",
                    "Question",
                    "Answer",
                ],
            )

        except Exception as exc:

            print(
                "  WARNING:",
                repr(exc),
            )

            continue


        mask = (
            ft["ID"]
            .astype(str)
            .str.strip()
            .isin(remaining_ids)
        )


        part = ft.loc[
            mask,
            [
                "ID",
                "General Category",
                "Category",
                "Question",
                "Answer",
            ],
        ]


        if len(part):

            resolved_parts.append(
                part.copy()
            )

            print(
                "  matched:",
                len(part),
            )


        del ft
        gc.collect()


    # ------------------------------------------------------------------
    # VERIFY RESOLUTION
    # ------------------------------------------------------------------

    if not resolved_parts:

        raise RuntimeError(
            "Could not resolve any selected IDs "
            "from the original IMB source."
        )


    resolved_df = pd.concat(
        resolved_parts,
        ignore_index=True,
    )


    resolved_df[
        "ID_key"
    ] = (
        resolved_df["ID"]
        .astype(str)
        .str.strip()
    )


    resolved_df = (
        resolved_df
        .drop_duplicates(
            "ID_key"
        )
    )


    found_ids = set(
        resolved_df[
            "ID_key"
        ]
    )

    missing_ids = (
        selected_ids
        - found_ids
    )


    print()
    print(
        "Selected IDs:",
        len(selected_ids),
    )

    print(
        "Resolved IDs:",
        len(found_ids),
    )

    print(
        "Missing IDs :",
        len(missing_ids),
    )


    if missing_ids:

        print(
            "First missing IDs:",
            sorted(missing_ids)[:30],
        )

        raise RuntimeError(
            "Not all 500 audit IDs could be "
            "resolved from the source dataset."
        )


    # Keep original deterministic sample order.
    base = audit_df[
        [
            "ID_key",
            "_audit_rank",
        ]
    ].copy()


    audit_df = base.merge(
        resolved_df[
            [
                "ID_key",
                "ID",
                "General Category",
                "Category",
                "Question",
                "Answer",
            ]
        ],
        on="ID_key",
        how="left",
        validate="one_to_one",
    )


    del resolved_df
    del resolved_parts
    gc.collect()


# ======================================================================================
# FINAL QA VALIDATION
# ======================================================================================

print()
print("=" * 110)
print("FINAL QA VALIDATION")
print("=" * 110)


required_final_columns = [
    "ID",
    "General Category",
    "Category",
    "Question",
    "Answer",
]


missing_columns = [
    c
    for c in required_final_columns
    if c not in audit_df.columns
]


if missing_columns:

    raise RuntimeError(
        "Final audit dataframe is missing columns: "
        f"{missing_columns}"
    )


empty_questions = (
    audit_df[
        "Question"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


empty_answers = (
    audit_df[
        "Answer"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


print(
    "Records        :",
    len(audit_df),
)

print(
    "Unique IDs     :",
    audit_df[
        "ID"
    ].astype(str).nunique(),
)

print(
    "Empty Question :",
    int(empty_questions),
)

print(
    "Empty Answer   :",
    int(empty_answers),
)


if empty_questions or empty_answers:

    raise RuntimeError(
        "Unexpected empty Question/Answer "
        "in audit sample."
    )


# ======================================================================================
# FINAL DETERMINISTIC ORDER
# ======================================================================================

audit_df = (
    audit_df
    .sort_values(
        [
            "General Category",
            "_audit_rank",
        ]
    )
    .reset_index(drop=True)
)


audit_df.insert(
    0,
    "Audit_Row",
    range(
        1,
        len(audit_df) + 1,
    ),
)


# ======================================================================================
# ADD EMPTY REVIEW FIELDS
# ======================================================================================

audit_df[
    "Quality_Class"
] = ""

audit_df[
    "Clinical_Relevance"
] = ""

audit_df[
    "Completeness"
] = ""

audit_df[
    "Potential_Error"
] = ""

audit_df[
    "Notes"
] = ""


# ======================================================================================
# SAMPLE DISTRIBUTION
# ======================================================================================

selected_distribution = (
    audit_df[
        "General Category"
    ]
    .value_counts()
    .sort_index()
)


print()
print("=" * 110)
print("AUDIT SAMPLE VALIDATION")
print("=" * 110)

print(
    "Selected records:",
    len(audit_df),
)

print(
    "Unique IDs      :",
    audit_df[
        "ID"
    ].astype(str).nunique(),
)

print()
print(
    "Selected distribution:"
)

print(
    selected_distribution
)


# ======================================================================================
# FINGERPRINT
# ======================================================================================

fingerprint_material = "\n".join(
    audit_df[
        "ID"
    ]
    .astype(str)
    .tolist()
)


sample_sha256 = hashlib.sha256(
    fingerprint_material.encode(
        "utf-8"
    )
).hexdigest()


print()
print(
    "Audit sample SHA256:",
    sample_sha256,
)


# ======================================================================================
# FINAL OUTPUT COLUMNS
# ======================================================================================

audit_df = audit_df[
    [
        "Audit_Row",
        "ID",
        "General Category",
        "Category",
        "Question",
        "Answer",
        "Quality_Class",
        "Clinical_Relevance",
        "Completeness",
        "Potential_Error",
        "Notes",
        "_audit_rank",
    ]
]


# ======================================================================================
# SAVE
# ======================================================================================

audit_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8",
)


audit_df.to_parquet(
    OUTPUT_PARQUET,
    index=False,
)


metadata = {
    "protocol_version":
        "v07.01",

    "cell":
        "24_v2",

    "purpose":
        (
            "Deterministic proportional-stratified "
            "IMB reference quality audit sample"
        ),

    "seed":
        SEED,

    "audit_n":
        AUDIT_N,

    "sampling_population":
        "frozen TRAIN pool",

    "source_train_rows":
        int(
            len(sampling_df)
        ),

    "frozen_test_used":
        False,

    "dev_used":
        False,

    "validation_used":
        False,

    "stratification_variable":
        "General Category",

    "allocation_method":
        "proportional largest remainder",

    "ranking_method":
        "SHA256(seed|ID|General Category)",

    "sample_sha256":
        sample_sha256,

    "category_quotas": {
        str(
            row[
                "General Category"
            ]
        ):
        int(
            row[
                "Quota"
            ]
        )

        for _, row
        in category_table.iterrows()
    },

    "output_csv":
        str(OUTPUT_CSV),

    "output_parquet":
        str(OUTPUT_PARQUET),
}


with OUTPUT_META.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ======================================================================================
# DISPLAY FIRST 20
# ======================================================================================

print()
print("=" * 110)
print("FIRST 20 FROZEN AUDIT RECORDS")
print("=" * 110)

pd.set_option(
    "display.max_colwidth",
    160,
)


display(
    audit_df[
        [
            "Audit_Row",
            "ID",
            "General Category",
            "Category",
            "Question",
            "Answer",
        ]
    ].head(20)
)


# ======================================================================================
# COMPLETE
# ======================================================================================

print()
print("=" * 110)
print("CELL 24 v2 COMPLETE")
print("=" * 110)

print(
    "CSV      :",
    OUTPUT_CSV,
)

print(
    "Parquet  :",
    OUTPUT_PARQUET,
)

print(
    "Metadata :",
    OUTPUT_META,
)

print()

print(
    "Quality fields remain intentionally empty."
)

print(
    "NEXT: review quotas and SHA256 fingerprint "
    "before beginning reference-quality classification."
)

In [3]:
# ======================================================================================
# CELL 24C — STANDALONE FULL-TEXT DISPLAY OF FIRST 20 FROZEN AUDIT RECORDS
# ======================================================================================
#
# PURPOSE
# -------
# Display the COMPLETE Question and Answer text for Audit_Row 1–20 from the
# already frozen IMB Reference Quality Audit sample.
#
# The normal pandas/Jupyter dataframe display abbreviates long text with "...".
# That truncation is only visual, but it prevents a rigorous manual calibration
# of the A/B/C/D reference-quality taxonomy.
#
# IMPORTANT
# ---------
# - Completely standalone.
# - Cells 1–24 do NOT need to be rerun.
# - Reads the frozen CSV only.
# - Does NOT modify the audit sample.
# - Does NOT classify records.
# - Does NOT use GPU, OpenAI, RAGAS or Unsloth.
# - Safe to rerun at any time.
#
# ======================================================================================

from pathlib import Path
import pandas as pd

AUDIT_PATH = Path(
    "/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/"
    "reports/imb_reference_quality_audit_500.csv"
)

if not AUDIT_PATH.exists():
    raise FileNotFoundError(AUDIT_PATH)

df = pd.read_csv(AUDIT_PATH)

first20 = (
    df.loc[df["Audit_Row"].between(1, 20)]
    .sort_values("Audit_Row")
)

print("=" * 110)
print("CELL 24C — FIRST 20 FROZEN AUDIT RECORDS — FULL TEXT")
print("=" * 110)

for _, row in first20.iterrows():

    print()
    print("=" * 110)
    print(
        f"AUDIT ROW {int(row['Audit_Row'])} | "
        f"ID {row['ID']} | "
        f"{row['General Category']} | "
        f"{row['Category']}"
    )
    print("=" * 110)

    print("\nQUESTION:")
    print(str(row["Question"]).strip())

    print("\nANSWER:")
    print(str(row["Answer"]).strip())

print()
print("=" * 110)
print("CELL 24C COMPLETE")
print("=" * 110)
print("No data were modified.")

In [5]:
# ======================================================================================
# CELL 25 — STANDALONE AUTOMATIC IMB REFERENCE INTEGRITY & SEMANTIC CONSISTENCY AUDIT
#           FROZEN SAMPLE N=500 — NO CLINICAL GOLD-STANDARD JUDGMENT
# ======================================================================================
#
# EMCRAG Fine-Tuning Experimental Protocol v07.01
#
# PURPOSE
# -------
# Cells 23–24C showed that some IMB reference answers contain observable dataset-quality
# problems: semantic mismatch, corrupted/contaminated text, very weak/non-autonomous
# replies, and generic refusals.
#
# IMPORTANT METHODOLOGICAL CORRECTION
# -----------------------------------
# This cell DOES NOT ask the researcher to judge clinical correctness and DOES NOT create
# A/B/C/D "clinical quality" labels.
#
# The user is not assumed to be a medical expert, and an automatic model is not treated
# as a substitute clinical gold-standard reviewer.
#
# Cell 25 therefore performs a narrower and defensible DATASET-QUALITY SCREEN:
#
#   - frozen-sample integrity check;
#   - structural/text-integrity diagnostics;
#   - short/non-autonomous/generic-response pattern flags;
#   - automatic Question↔Answer semantic-consistency screening;
#   - descriptive anomaly flags and summaries.
#
# These flags are SCREENING SIGNALS, not diagnoses and not proof that an answer is
# medically correct or incorrect.
#
# FAST-PATH ROLE IN THE EXPERIMENT
# --------------------------------
# This audit is intentionally lightweight. It documents the limitation discovered in IMB
# and then allows the main experiment to continue rapidly toward the four systems:
#
#   M0 = base MedGemma standalone
#   M1 = selected fine-tuned MedGemma standalone
#   R0 = frozen EMCRAG v06.11 + base MedGemma
#   R1 = frozen EMCRAG v06.11 + selected fine-tuned MedGemma
#
# No manual review of 480 records is required.
#
# INPUT
# -----
# /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/
#     imb_reference_quality_audit_500.csv
#
# FROZEN INPUT SHA256
# -------------------
# 0659115a65e17e1fdbbbe0812f589e4e7a5caee3c3ac679095c62d574719b4a5
#
# OUTPUTS
# -------
# reports/cell25_automatic_reference_audit_500.csv
# reports/cell25_automatic_reference_audit_500.parquet
# reports/cell25_automatic_reference_audit_summary.csv
# reports/cell25_automatic_reference_audit_by_category.csv
# reports/cell25_automatic_reference_audit_metadata.json
# reports/cell25_automatic_reference_audit_report.md
#
# SEMANTIC MODEL
# --------------
# Preferred local model:
#
#   sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
#
# This is already consistent with the EMCRAG retrieval stack.
#
# The cell first attempts to load the SentenceTransformer locally. If that is unavailable,
# it falls back to a deterministic character n-gram TF-IDF cosine screen.
#
# The fallback remains useful for detecting gross mismatch but is explicitly recorded in
# metadata and must not be interpreted as equivalent to neural semantic similarity.
#
# AUTOMATIC FLAGS
# ---------------
# VERY_SHORT
#     Answer has < 12 lexical words.
#
# NON_AUTONOMOUS
#     Short answer whose main content is agreement/deference to another clinician.
#
# GENERIC_REFUSAL
#     Short answer dominated by a generic "cannot provide diagnosis/consultation" pattern.
#
# TEXT_CORRUPTION
#     Obvious text-integrity artifacts such as mojibake, replacement characters,
#     model-template tokens, or known malformed fragments.
#
# SEMANTIC_BOTTOM_10
#     Question↔Answer semantic score is in the bottom 10% of the frozen sample.
#     This is a distributional screening flag only.
#
# SEMANTIC_EXTREME_LOW
#     Score is in the bottom 5% of the sample.
#
# LEXICAL_BOTTOM_10
#     Question↔Answer lexical Jaccard is in the bottom 10%.
#
# LIKELY_MISMATCH
#     Conservative combined flag:
#         SEMANTIC_EXTREME_LOW AND LEXICAL_BOTTOM_10
#
# AUTOMATIC_STATUS
# ----------------
# HIGH_FLAG
#     TEXT_CORRUPTION OR NON_AUTONOMOUS OR LIKELY_MISMATCH
#
# MODERATE_FLAG
#     Not HIGH_FLAG, but at least one of:
#         VERY_SHORT
#         GENERIC_REFUSAL
#         SEMANTIC_BOTTOM_10
#
# NO_AUTOMATIC_FLAG
#     None of the above screening conditions triggered.
#
# NOTE:
# NO_AUTOMATIC_FLAG does NOT mean "clinically correct".
# HIGH_FLAG does NOT mean "clinically wrong".
#
# ======================================================================================


from pathlib import Path
import hashlib
import inspect
import json
import math
import re
import sys
from collections import Counter

import numpy as np
import pandas as pd


# ======================================================================================
# CONFIG
# ======================================================================================

PROTOCOL_VERSION = "v07.01"
CELL_VERSION = "25_automatic_v2"

EXPECTED_N = 500
EXPECTED_SHA256 = (
    "0659115a65e17e1fdbbbe0812f589e4e7a5caee3c3ac679095c62d574719b4a5"
)

BASE_DIR = Path(
    "/home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01"
)

REPORT_DIR = BASE_DIR / "reports"

INPUT_CSV = (
    REPORT_DIR
    / "imb_reference_quality_audit_500.csv"
)

OUTPUT_CSV = (
    REPORT_DIR
    / "cell25_automatic_reference_audit_500.csv"
)

OUTPUT_PARQUET = (
    REPORT_DIR
    / "cell25_automatic_reference_audit_500.parquet"
)

SUMMARY_CSV = (
    REPORT_DIR
    / "cell25_automatic_reference_audit_summary.csv"
)

CATEGORY_CSV = (
    REPORT_DIR
    / "cell25_automatic_reference_audit_by_category.csv"
)

META_JSON = (
    REPORT_DIR
    / "cell25_automatic_reference_audit_metadata.json"
)

REPORT_MD = (
    REPORT_DIR
    / "cell25_automatic_reference_audit_report.md"
)

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

EMBED_BATCH_SIZE = 64


# ======================================================================================
# TEXT HELPERS
# ======================================================================================

TOKEN_RE = re.compile(
    r"\b[\wÀ-ÿ]+\b",
    flags=re.UNICODE,
)

# Strong, observable text-integrity artifacts only.
CORRUPTION_PATTERNS = [
    r"�",
    r"Ã[\x80-\xBF]",
    r"Â[\x80-\xBF]",
    r"<unused\d+>",
    r"<\|[^>]+\|>",
    r"\\u[0-9a-fA-F]{4}",
    r"\bi['’]?ll tuo arrivederci\b",
]

NON_AUTONOMOUS_PATTERNS = [
    r"\bcondivido\b.{0,40}\b(collega|indicazione|posizione)\b",
    r"\bsono (?:in )?accordo\b.{0,40}\b(collega|indicazione|posizione)\b",
    r"\bconcordo\b.{0,40}\b(collega|indicazione|posizione)\b",
    r"\bcome (?:già )?(?:detto|indicato) dal collega\b",
]

GENERIC_REFUSAL_PATTERNS = [
    r"\bnon posso fornire (?:una )?(?:diagnosi|consulenza|consulto|parere medico)\b",
    r"\bnon è possibile fornire (?:una )?(?:diagnosi|consulenza|risposta univoca)\b",
    r"\bnon posso (?:fare|formulare) (?:una )?diagnosi\b",
]

def lexical_tokens(text):
    return [
        token.lower()
        for token in TOKEN_RE.findall(
            str(text or "")
        )
    ]


def lexical_jaccard(question, answer):
    q = set(lexical_tokens(question))
    a = set(lexical_tokens(answer))

    if not q and not a:
        return 1.0

    if not q or not a:
        return 0.0

    return len(q & a) / len(q | a)


def regex_any(text, patterns):
    s = str(text or "")
    return any(
        re.search(
            pattern,
            s,
            flags=re.IGNORECASE | re.DOTALL,
        )
        is not None
        for pattern in patterns
    )


# ======================================================================================
# START
# ======================================================================================

print("=" * 112)
print("CELL 25 — AUTOMATIC IMB REFERENCE INTEGRITY & SEMANTIC CONSISTENCY AUDIT")
print("=" * 112)

print("Protocol version :", PROTOCOL_VERSION)
print("Cell version     :", CELL_VERSION)
print("Expected N       :", EXPECTED_N)
print("Expected SHA256  :", EXPECTED_SHA256)
print("Input            :", INPUT_CSV)
print()


# ======================================================================================
# LOAD + VERIFY FROZEN SAMPLE
# ======================================================================================

if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Frozen audit sample not found:\n{INPUT_CSV}"
    )

df = pd.read_csv(
    INPUT_CSV,
    dtype={
        "ID": str,
    },
)

required_columns = {
    "Audit_Row",
    "ID",
    "General Category",
    "Category",
    "Question",
    "Answer",
}

missing = required_columns - set(df.columns)

if missing:
    raise RuntimeError(
        "Missing required columns: "
        f"{sorted(missing)}"
    )

df["Audit_Row"] = pd.to_numeric(
    df["Audit_Row"],
    errors="raise",
).astype(int)

df["ID"] = (
    df["ID"]
    .astype(str)
    .str.strip()
)

df = (
    df
    .sort_values("Audit_Row")
    .reset_index(drop=True)
)

if len(df) != EXPECTED_N:
    raise RuntimeError(
        f"Expected {EXPECTED_N} records; found {len(df)}."
    )

if df["ID"].nunique() != EXPECTED_N:
    raise RuntimeError(
        "The frozen sample does not contain 500 unique IDs."
    )

fingerprint_material = "\n".join(
    df["ID"].tolist()
)

actual_sha256 = hashlib.sha256(
    fingerprint_material.encode("utf-8")
).hexdigest()

print("=" * 112)
print("FROZEN SAMPLE VERIFICATION")
print("=" * 112)

print("Actual N      :", len(df))
print("Unique IDs    :", df["ID"].nunique())
print("Actual SHA256 :", actual_sha256)
print("Fingerprint OK:", actual_sha256 == EXPECTED_SHA256)

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "STOP: frozen sample fingerprint mismatch."
    )


# ======================================================================================
# STRUCTURAL / TEXT-INTEGRITY FEATURES
# ======================================================================================

print()
print("=" * 112)
print("STRUCTURAL AND TEXT-INTEGRITY FEATURES")
print("=" * 112)

df["Question_Words"] = (
    df["Question"]
    .fillna("")
    .map(
        lambda x: len(
            lexical_tokens(x)
        )
    )
)

df["Answer_Words"] = (
    df["Answer"]
    .fillna("")
    .map(
        lambda x: len(
            lexical_tokens(x)
        )
    )
)

df["Answer_Chars"] = (
    df["Answer"]
    .fillna("")
    .astype(str)
    .str.len()
)

df["QA_Lexical_Jaccard"] = [
    lexical_jaccard(q, a)
    for q, a in zip(
        df["Question"],
        df["Answer"],
    )
]

df["VERY_SHORT"] = (
    df["Answer_Words"]
    < 12
)

df["TEXT_CORRUPTION"] = (
    df["Answer"]
    .map(
        lambda x:
            regex_any(
                x,
                CORRUPTION_PATTERNS,
            )
    )
)

df["NON_AUTONOMOUS"] = [
    (
        words <= 40
        and regex_any(
            answer,
            NON_AUTONOMOUS_PATTERNS,
        )
    )
    for answer, words in zip(
        df["Answer"],
        df["Answer_Words"],
    )
]

df["GENERIC_REFUSAL"] = [
    (
        words <= 60
        and regex_any(
            answer,
            GENERIC_REFUSAL_PATTERNS,
        )
    )
    for answer, words in zip(
        df["Answer"],
        df["Answer_Words"],
    )
]

print(
    "Very-short answers     :",
    int(df["VERY_SHORT"].sum()),
)

print(
    "Text-corruption flags  :",
    int(df["TEXT_CORRUPTION"].sum()),
)

print(
    "Non-autonomous flags   :",
    int(df["NON_AUTONOMOUS"].sum()),
)

print(
    "Generic-refusal flags  :",
    int(df["GENERIC_REFUSAL"].sum()),
)


# ======================================================================================
# SEMANTIC CONSISTENCY
# ======================================================================================

print()
print("=" * 112)
print("QUESTION↔ANSWER SEMANTIC CONSISTENCY")
print("=" * 112)

semantic_method = None
semantic_model = None
semantic_device = None
semantic_error = None

questions = (
    df["Question"]
    .fillna("")
    .astype(str)
    .tolist()
)

answers = (
    df["Answer"]
    .fillna("")
    .astype(str)
    .tolist()
)

# ------------------------------------------------------------------
# Preferred: local SentenceTransformer.
# ------------------------------------------------------------------

try:
    import torch
    from sentence_transformers import SentenceTransformer

    semantic_device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    constructor_signature = inspect.signature(
        SentenceTransformer
    )

    st_kwargs = {
        "device":
            semantic_device,
    }

    if (
        "local_files_only"
        in constructor_signature.parameters
    ):
        st_kwargs[
            "local_files_only"
        ] = True

    semantic_model = SentenceTransformer(
        EMBEDDING_MODEL_NAME,
        **st_kwargs,
    )

    q_emb = semantic_model.encode(
        questions,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    a_emb = semantic_model.encode(
        answers,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    semantic_scores = np.sum(
        q_emb * a_emb,
        axis=1,
    )

    semantic_method = (
        "sentence_transformers_cosine"
    )

    print(
        "Semantic method : SentenceTransformer cosine"
    )

    print(
        "Model           :",
        EMBEDDING_MODEL_NAME,
    )

    print(
        "Device          :",
        semantic_device,
    )

except Exception as exc:

    semantic_error = repr(exc)

    print(
        "WARNING: local SentenceTransformer unavailable."
    )

    print(
        "Reason:",
        semantic_error,
    )

    print(
        "Falling back to deterministic character n-gram TF-IDF cosine."
    )

    from sklearn.feature_extraction.text import TfidfVectorizer

    # Joint fit preserves a common vector space.
    joint_text = questions + answers

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        max_features=80_000,
        sublinear_tf=True,
        norm="l2",
    )

    matrix = vectorizer.fit_transform(
        joint_text
    )

    q_matrix = matrix[:EXPECTED_N]
    a_matrix = matrix[EXPECTED_N:]

    semantic_scores = np.asarray(
        q_matrix.multiply(
            a_matrix
        ).sum(axis=1)
    ).ravel()

    semantic_method = (
        "char_ngram_tfidf_cosine_fallback"
    )

    semantic_device = "cpu"

df["Semantic_Score"] = (
    semantic_scores.astype(float)
)


# ======================================================================================
# DISTRIBUTIONAL THRESHOLDS
# ======================================================================================

semantic_q05 = float(
    df["Semantic_Score"].quantile(0.05)
)

semantic_q10 = float(
    df["Semantic_Score"].quantile(0.10)
)

semantic_q25 = float(
    df["Semantic_Score"].quantile(0.25)
)

semantic_q50 = float(
    df["Semantic_Score"].quantile(0.50)
)

semantic_q75 = float(
    df["Semantic_Score"].quantile(0.75)
)

semantic_iqr = (
    semantic_q75
    - semantic_q25
)

semantic_lower_fence = float(
    semantic_q25
    - 1.5 * semantic_iqr
)

lexical_q10 = float(
    df["QA_Lexical_Jaccard"].quantile(0.10)
)

df["SEMANTIC_BOTTOM_10"] = (
    df["Semantic_Score"]
    <= semantic_q10
)

df["SEMANTIC_EXTREME_LOW"] = (
    df["Semantic_Score"]
    <= semantic_q05
)

df["SEMANTIC_IQR_OUTLIER"] = (
    df["Semantic_Score"]
    < semantic_lower_fence
)

df["LEXICAL_BOTTOM_10"] = (
    df["QA_Lexical_Jaccard"]
    <= lexical_q10
)

# Conservative likely mismatch:
# both neural/distributional semantic extreme-low AND lexical extreme-low.
df["LIKELY_MISMATCH"] = (
    df["SEMANTIC_EXTREME_LOW"]
    & df["LEXICAL_BOTTOM_10"]
)

print(
    "Semantic score q05     :",
    f"{semantic_q05:.6f}",
)

print(
    "Semantic score q10     :",
    f"{semantic_q10:.6f}",
)

print(
    "Semantic score median  :",
    f"{semantic_q50:.6f}",
)

print(
    "Semantic IQR lower fence:",
    f"{semantic_lower_fence:.6f}",
)

print(
    "Lexical Jaccard q10    :",
    f"{lexical_q10:.6f}",
)

print(
    "Likely mismatch flags  :",
    int(df["LIKELY_MISMATCH"].sum()),
)


# ======================================================================================
# AUTOMATIC STATUS
# ======================================================================================

def build_flag_list(row):

    ordered_flags = [
        "TEXT_CORRUPTION",
        "NON_AUTONOMOUS",
        "GENERIC_REFUSAL",
        "VERY_SHORT",
        "SEMANTIC_BOTTOM_10",
        "SEMANTIC_EXTREME_LOW",
        "SEMANTIC_IQR_OUTLIER",
        "LEXICAL_BOTTOM_10",
        "LIKELY_MISMATCH",
    ]

    return ";".join(
        flag
        for flag in ordered_flags
        if bool(row[flag])
    )


df["Flag_List"] = (
    df.apply(
        build_flag_list,
        axis=1,
    )
)


def automatic_status(row):

    if (
        row["TEXT_CORRUPTION"]
        or row["NON_AUTONOMOUS"]
        or row["LIKELY_MISMATCH"]
    ):
        return "HIGH_FLAG"

    if (
        row["VERY_SHORT"]
        or row["GENERIC_REFUSAL"]
        or row["SEMANTIC_BOTTOM_10"]
    ):
        return "MODERATE_FLAG"

    return "NO_AUTOMATIC_FLAG"


df["Automatic_Status"] = (
    df.apply(
        automatic_status,
        axis=1,
    )
)

df["Any_Automatic_Flag"] = (
    df["Automatic_Status"]
    != "NO_AUTOMATIC_FLAG"
)


# ======================================================================================
# SUMMARY
# ======================================================================================

status_order = [
    "NO_AUTOMATIC_FLAG",
    "MODERATE_FLAG",
    "HIGH_FLAG",
]

status_counts = (
    df["Automatic_Status"]
    .value_counts()
    .reindex(
        status_order,
        fill_value=0,
    )
)

summary_rows = []

for status in status_order:

    count = int(
        status_counts[
            status
        ]
    )

    summary_rows.append(
        {
            "Metric":
                f"Automatic_Status::{status}",

            "Count":
                count,

            "Rate":
                count / EXPECTED_N,
        }
    )

flag_columns = [
    "VERY_SHORT",
    "TEXT_CORRUPTION",
    "NON_AUTONOMOUS",
    "GENERIC_REFUSAL",
    "SEMANTIC_BOTTOM_10",
    "SEMANTIC_EXTREME_LOW",
    "SEMANTIC_IQR_OUTLIER",
    "LEXICAL_BOTTOM_10",
    "LIKELY_MISMATCH",
]

for flag in flag_columns:

    count = int(
        df[flag].sum()
    )

    summary_rows.append(
        {
            "Metric":
                f"Flag::{flag}",

            "Count":
                count,

            "Rate":
                count / EXPECTED_N,
        }
    )

summary_df = pd.DataFrame(
    summary_rows
)

category_summary = (
    df
    .groupby(
        "General Category"
    )
    .agg(
        N=(
            "ID",
            "size",
        ),

        HIGH_FLAG=(
            "Automatic_Status",
            lambda s:
                int(
                    (
                        s
                        == "HIGH_FLAG"
                    ).sum()
                ),
        ),

        MODERATE_FLAG=(
            "Automatic_Status",
            lambda s:
                int(
                    (
                        s
                        == "MODERATE_FLAG"
                    ).sum()
                ),
        ),

        NO_AUTOMATIC_FLAG=(
            "Automatic_Status",
            lambda s:
                int(
                    (
                        s
                        == "NO_AUTOMATIC_FLAG"
                    ).sum()
                ),
        ),

        LIKELY_MISMATCH=(
            "LIKELY_MISMATCH",
            "sum",
        ),

        TEXT_CORRUPTION=(
            "TEXT_CORRUPTION",
            "sum",
        ),

        NON_AUTONOMOUS=(
            "NON_AUTONOMOUS",
            "sum",
        ),

        VERY_SHORT=(
            "VERY_SHORT",
            "sum",
        ),

        Median_Semantic_Score=(
            "Semantic_Score",
            "median",
        ),
    )
    .reset_index()
)

for col in [
    "HIGH_FLAG",
    "MODERATE_FLAG",
    "NO_AUTOMATIC_FLAG",
    "LIKELY_MISMATCH",
    "TEXT_CORRUPTION",
    "NON_AUTONOMOUS",
    "VERY_SHORT",
]:
    category_summary[col] = (
        category_summary[col]
        .astype(int)
    )

category_summary[
    "HIGH_FLAG_Rate"
] = (
    category_summary[
        "HIGH_FLAG"
    ]
    / category_summary[
        "N"
    ]
)


# ======================================================================================
# CALIBRATION SANITY CHECK — OBSERVATION ONLY
# ======================================================================================
#
# These rows were already inspected in Cell 24C.
# They are NOT used to set thresholds and are NOT included as labels.
# This table only checks whether the automatic screen reacts to known examples.
# ======================================================================================

known_observation_rows = [
    2,
    13,
    17,
    18,
    20,
]

sanity = (
    df.loc[
        df["Audit_Row"]
        .isin(
            known_observation_rows
        ),
        [
            "Audit_Row",
            "ID",
            "Semantic_Score",
            "QA_Lexical_Jaccard",
            "VERY_SHORT",
            "TEXT_CORRUPTION",
            "NON_AUTONOMOUS",
            "GENERIC_REFUSAL",
            "SEMANTIC_BOTTOM_10",
            "SEMANTIC_EXTREME_LOW",
            "LIKELY_MISMATCH",
            "Automatic_Status",
            "Flag_List",
        ],
    ]
    .sort_values(
        "Audit_Row"
    )
)


# ======================================================================================
# SAVE
# ======================================================================================

df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8",
)

df.to_parquet(
    OUTPUT_PARQUET,
    index=False,
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

category_summary.to_csv(
    CATEGORY_CSV,
    index=False,
)


# ======================================================================================
# METADATA
# ======================================================================================

metadata = {
    "protocol_version":
        PROTOCOL_VERSION,

    "cell_version":
        CELL_VERSION,

    "purpose":
        (
            "Automatic non-clinical integrity and semantic-consistency "
            "screen of frozen IMB reference answers"
        ),

    "frozen_sample_n":
        EXPECTED_N,

    "frozen_sample_sha256":
        actual_sha256,

    "clinical_gold_standard_judgment":
        False,

    "manual_480_record_review_required":
        False,

    "semantic_method":
        semantic_method,

    "semantic_model":
        (
            EMBEDDING_MODEL_NAME
            if semantic_method
            == "sentence_transformers_cosine"
            else None
        ),

    "semantic_device":
        semantic_device,

    "sentence_transformer_error":
        semantic_error,

    "thresholds":
        {
            "semantic_q05":
                semantic_q05,

            "semantic_q10":
                semantic_q10,

            "semantic_q25":
                semantic_q25,

            "semantic_q50":
                semantic_q50,

            "semantic_q75":
                semantic_q75,

            "semantic_iqr_lower_fence":
                semantic_lower_fence,

            "lexical_jaccard_q10":
                lexical_q10,
        },

    "interpretation":
        (
            "Automatic flags are dataset-quality screening signals only; "
            "they do not establish medical correctness or incorrectness."
        ),

    "outputs":
        {
            "record_level_csv":
                str(OUTPUT_CSV),

            "record_level_parquet":
                str(OUTPUT_PARQUET),

            "summary_csv":
                str(SUMMARY_CSV),

            "category_csv":
                str(CATEGORY_CSV),

            "report_md":
                str(REPORT_MD),
        },
}

with META_JSON.open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ======================================================================================
# HUMAN-READABLE REPORT
# ======================================================================================

high_n = int(
    (
        df["Automatic_Status"]
        == "HIGH_FLAG"
    ).sum()
)

moderate_n = int(
    (
        df["Automatic_Status"]
        == "MODERATE_FLAG"
    ).sum()
)

clean_n = int(
    (
        df["Automatic_Status"]
        == "NO_AUTOMATIC_FLAG"
    ).sum()
)

report_text = f"""# Cell 25 — Automatic IMB Reference Integrity & Semantic Consistency Audit

Protocol: {PROTOCOL_VERSION}

Frozen sample N: {EXPECTED_N}

Frozen sample SHA256:

`{actual_sha256}`

## Method

This is an automatic **dataset-quality screening audit**, not a clinical correctness assessment.

Semantic method: `{semantic_method}`

Semantic model: `{EMBEDDING_MODEL_NAME if semantic_method == "sentence_transformers_cosine" else "TF-IDF fallback"}`

## Automatic status

- NO_AUTOMATIC_FLAG: {clean_n} / {EXPECTED_N} ({clean_n/EXPECTED_N:.1%})
- MODERATE_FLAG: {moderate_n} / {EXPECTED_N} ({moderate_n/EXPECTED_N:.1%})
- HIGH_FLAG: {high_n} / {EXPECTED_N} ({high_n/EXPECTED_N:.1%})

## Key technical flags

- VERY_SHORT: {int(df["VERY_SHORT"].sum())}
- TEXT_CORRUPTION: {int(df["TEXT_CORRUPTION"].sum())}
- NON_AUTONOMOUS: {int(df["NON_AUTONOMOUS"].sum())}
- GENERIC_REFUSAL: {int(df["GENERIC_REFUSAL"].sum())}
- LIKELY_MISMATCH: {int(df["LIKELY_MISMATCH"].sum())}

## Interpretation

These counts quantify automatically detectable **reference-integrity and semantic-consistency signals**.

They must not be reported as a medical error rate.

`NO_AUTOMATIC_FLAG` does not mean clinically correct.

`HIGH_FLAG` does not mean clinically wrong.

The scientific implication is narrower: IMB reference answers should not be assumed a priori to be uniformly suitable clinical gold-standard answers. Agreement with IMB reference text is therefore best interpreted as a dataset-alignment signal unless supplemented by an independent clinically valid evaluation source.

## Experimental consequence

No 480-record manual clinical review is required.

After this Cell 25 audit is recorded, the primary experimental workflow returns to the fine-tuning learning curve and construction of M0/M1/R0/R1.
"""

REPORT_MD.write_text(
    report_text,
    encoding="utf-8",
)


# ======================================================================================
# CONSOLE OUTPUT
# ======================================================================================

print()
print("=" * 112)
print("AUTOMATIC AUDIT SUMMARY")
print("=" * 112)

display(
    summary_df
)

print()
print("=" * 112)
print("CATEGORY SUMMARY")
print("=" * 112)

display(
    category_summary
)

print()
print("=" * 112)
print("CALIBRATION SANITY CHECK — OBSERVATION ONLY")
print("=" * 112)

display(
    sanity
)

print()
print("=" * 112)
print("CELL 25 COMPLETE")
print("=" * 112)

print("Semantic method :", semantic_method)
print("Record CSV      :", OUTPUT_CSV)
print("Record Parquet  :", OUTPUT_PARQUET)
print("Summary CSV     :", SUMMARY_CSV)
print("Category CSV    :", CATEGORY_CSV)
print("Metadata        :", META_JSON)
print("Report          :", REPORT_MD)

print()
print(
    "No manual 480-record review is required."
)

print(
    "NEXT: record this audit result, then return to the FT learning curve "
    "starting with FT-10K from the original base MedGemma."
)


CELL 25 — AUTOMATIC IMB REFERENCE INTEGRITY & SEMANTIC CONSISTENCY AUDIT
Protocol version : v07.01
Cell version     : 25_automatic_v2
Expected N       : 500
Expected SHA256  : 0659115a65e17e1fdbbbe0812f589e4e7a5caee3c3ac679095c62d574719b4a5
Input            : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/imb_reference_quality_audit_500.csv

FROZEN SAMPLE VERIFICATION
Actual N      : 500
Unique IDs    : 500
Actual SHA256 : 0659115a65e17e1fdbbbe0812f589e4e7a5caee3c3ac679095c62d574719b4a5
Fingerprint OK: True

STRUCTURAL AND TEXT-INTEGRITY FEATURES
Very-short answers     : 29
Text-corruption flags  : 1
Non-autonomous flags   : 2
Generic-refusal flags  : 7

QUESTION↔ANSWER SEMANTIC CONSISTENCY


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Semantic method : SentenceTransformer cosine
Model           : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Device          : cuda
Semantic score q05     : 0.190751
Semantic score q10     : 0.237697
Semantic score median  : 0.477630
Semantic IQR lower fence: 0.034745
Lexical Jaccard q10    : 0.036878
Likely mismatch flags  : 12

AUTOMATIC AUDIT SUMMARY


,Metric,Count,Rate
0,Automatic_Status::NO_AUTOMATIC_FLAG,429,0.858
1,Automatic_Status::MODERATE_FLAG,57,0.114
2,Automatic_Status::HIGH_FLAG,14,0.028
3,Flag::VERY_SHORT,29,0.058
4,Flag::TEXT_CORRUPTION,1,0.002
5,Flag::NON_AUTONOMOUS,2,0.004
6,Flag::GENERIC_REFUSAL,7,0.014
7,Flag::SEMANTIC_BOTTOM_10,50,0.100
8,Flag::SEMANTIC_EXTREME_LOW,25,0.050
9,Flag::SEMANTIC_IQR_OUTLIER,1,0.002



CATEGORY SUMMARY


,General Category,N,HIGH_FLAG,MODERATE_FLAG,NO_AUTOMATIC_FLAG,LIKELY_MISMATCH,TEXT_CORRUPTION,NON_AUTONOMOUS,VERY_SHORT,Median_Semantic_Score,HIGH_FLAG_Rate
0,"Cardiologia, Sistema Circolatorio ed Ematologia",52,5,7,40,3,1,1,4,0.435390,0.096154
1,"Dermatologia, Allergie e Estetica",31,0,5,26,0,0,0,1,0.487146,0.000000
2,Gastroenterologia e Salute Digestiva,67,3,8,56,3,0,0,5,0.457350,0.044776
3,Ginecologia e Salute Femminile,42,0,8,34,0,0,0,2,0.468172,0.000000
4,Medicina Generale e Chirurgia Generale,56,2,7,47,2,0,1,7,0.425140,0.035714
5,Neurologia,30,1,3,26,1,0,0,2,0.521698,0.033333
6,"Oculistica, Otorinolaringoiatria, Odontoiatria e Pneumologia",54,0,3,51,0,0,0,1,0.515728,0.000000
7,Ortopedia e Sistema Muscoloscheletrico,32,1,1,30,1,0,0,1,0.490048,0.031250
8,Salute Mentale,66,2,7,57,2,0,0,4,0.498564,0.030303
9,"Urologia, Andrologia e Salute Maschile",70,0,8,62,0,0,0,2,0.461412,0.000000



CALIBRATION SANITY CHECK — OBSERVATION ONLY


,Audit_Row,ID,Semantic_Score,QA_Lexical_Jaccard,VERY_SHORT,TEXT_CORRUPTION,NON_AUTONOMOUS,GENERIC_REFUSAL,SEMANTIC_BOTTOM_10,SEMANTIC_EXTREME_LOW,LIKELY_MISMATCH,Automatic_Status,Flag_List
1,2,422073,0.589038,0.063158,False,True,False,True,False,False,False,HIGH_FLAG,TEXT_CORRUPTION;GENERIC_REFUSAL
12,13,449925,0.284223,0.068182,False,False,False,True,False,False,False,MODERATE_FLAG,GENERIC_REFUSAL
16,17,779864,0.298053,0.055556,False,False,True,False,False,False,False,HIGH_FLAG,NON_AUTONOMOUS
17,18,479997,0.286571,0.051020,False,False,False,False,False,False,False,NO_AUTOMATIC_FLAG,
19,20,155981,0.601603,0.086667,False,False,False,False,False,False,False,NO_AUTOMATIC_FLAG,



CELL 25 COMPLETE
Semantic method : sentence_transformers_cosine
Record CSV      : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell25_automatic_reference_audit_500.csv
Record Parquet  : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell25_automatic_reference_audit_500.parquet
Summary CSV     : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell25_automatic_reference_audit_summary.csv
Category CSV    : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell25_automatic_reference_audit_by_category.csv
Metadata        : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell25_automatic_reference_audit_metadata.json
Report          : /home/eugrug/Project_CRAG_MedGemma/fine_tuning/v07_01/reports/cell25_automatic_reference_audit_report.md

No manual 480-record review is required.
NEXT: record this audit result, then return to the FT learning curve starting with FT-10K from the original base MedGemma.
